# RAGAS y scorecard final del tutor agéntico

### Sesión 10 · Módulo 3 — Evaluar por dentro: contexto, fidelidad y relevancia (4/4)

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

### Por qué no basta el harness

El harness dice **si** el tutor acertó. No dice **por qué**:

- ¿El contexto recuperado era el correcto?
- ¿La respuesta se apoya en lo que el sistema observó, o el modelo ya lo sabía?

Una respuesta puede acertar por la razón equivocada o fallar por un contexto pobre.
RAGAS separa esos casos y **complementa** al harness, no lo reemplaza. La entrega
M3 pide los dos juntos.

### Las cuatro métricas, adaptadas a este dominio

| Métrica | Pregunta | Adaptación para un agente con herramientas |
|---|---|---|
| **faithfulness** | ¿Cada afirmación se apoya en lo observado? | Lo observado = documentos **y resultados de herramientas**. Un cálculo hecho "de cabeza" no es fiel aunque sea correcto. |
| **context precision** | ¿Los chunks útiles quedaron arriba? | Solo en casos que requieren documentos. |
| **context recall** | ¿Llegó todo lo necesario? | Ídem. |
| **answer relevancy** | ¿La respuesta va al grano? | Sin cambios. |

### Dos decisiones que hay que declarar

1. **Evaluador de 7B** (Qwen2.5-7B-Instruct en 4 bits), no el juez de 1.5B de M2,
   que tuvo kappa 0.286. El juez de M2 sigue intacto en el harness, para no romper la
   comparabilidad.
2. **Cada métrica tiene una versión objetiva** (claves de oro y números respaldados)
   que **calibra** al evaluador, igual que en M2 se calibró el juez contra la verdad
   numérica. Si el evaluador no pasa la calibración, las conclusiones se apoyan en las
   objetivas.

> **Sobre la librería `ragas`.** Hace esto mismo por dentro, pero por defecto llama a
> OpenAI. Aquí se implementa "a mano" (como en el Lab C de S10) con un evaluador local,
> lo que además permite adaptar la faithfulness a las herramientas. Al final queda la
> celda de referencia con la librería.

## 2 · Objetivos

1. Calcular las cuatro métricas de RAGAS sobre las respuestas de C0–C5 (y C5-FT).
2. **Calibrar** el evaluador contra las versiones objetivas.
3. Detectar los **aciertos por la razón equivocada**.
4. Diagnosticar los fallos de conocimiento: ¿retrieval o generación?
5. Construir el **scorecard final de M3** (harness + agente + RAGAS) y registrarlo en W&B.

## 0 · Preparación del entorno

> **GPU T4.** Solo se carga el evaluador de 7B (~5.5 GB en 4 bits) y MiniLM. El
> cálculo se guarda incrementalmente en `resultados/ragas_s10.jsonl` y se retoma si
> Colab se desconecta.

In [1]:
%pip install -q "transformers>=4.44" "accelerate>=0.33" bitsandbytes sentence-transformers pandas scipy wandb
print("Listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 83.3 MB/s eta 0:00:00
Listo.


In [2]:
import os, random, re, time, json
import numpy as np
import torch
import transformers
from pathlib import Path

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | {DEVICE}")

transformers 5.16.1 | torch 2.11.0+cu128 | cuda


### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [3]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

Mounted at /content/drive
En Colab           : True
Directorio de trabajo: /content/drive/MyDrive/ProyectoIA
Checkpoints en     : /content/salidas


---
## 3 · Los datos

## 3.1 · El eval set

El eval set **no es el corpus de entrenamiento**. Es un conjunto nuevo, escrito
a mano, que cumple los cuatro criterios de S05:

| Criterio | Cómo se cumple aquí |
|---|---|
| **Representativo** | Problemas que un estudiante real plantearía, con enunciados en lenguaje natural. |
| **Con salida esperada** | Cada caso trae la respuesta correcta y un `criterio` explícito de qué la hace correcta. |
| **Cubre casos difíciles** | Es el eje del diseño: multipaso, distractores, porcentajes encadenados, proporcionalidad inversa, redondeo contextual. |
| **No contaminado** | Ningún caso proviene de `math_tutor_dataset.jsonl`. El generador lo verifica automáticamente y falla si hay solapamiento. |

**Por qué hizo falta uno nuevo.** En M1 el modelo *sin entrenar* ya resolvía el
90.91% del conjunto de validación. Un eval set con ese nivel de dificultad no
puede discriminar entre sistemas: todo se ve bien. Este está construido para
que el modelo falle, porque un eval set que no reta al sistema no informa nada.

### Los tres bloques

- **A · CÁLCULO (12 casos)** — mide **habilidad**. Cada uno tiene un modo de
  fallo documentado en su `criterio` (por ejemplo, sumar descuentos sucesivos
  en lugar de encadenarlos).
- **B · CONOCIMIENTO (4 casos)** — mide **conocimiento institucional** que
  ningún modelo puede tener en sus pesos. El baseline debería abstenerse aquí;
  si en cambio inventa una cifra con seguridad, es una alucinación y el
  scorecard la registra. **Este bloque es el que motiva el RAG de M3.**
- **C · ADVERSARIAL (5 casos)** — mide **robustez**, siguiendo el red-teaming
  de S06: premisa falsa, indefinición matemática, datos insuficientes, fuera de
  dominio y mantenimiento del rol.

Supera de largo el mínimo de M2 (≥10 gold + ≥2 adversariales): 16 gold y 5
adversariales.

In [4]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_EVAL = Path("data/eval_set_m2.jsonl")
SHA_EVAL  = "ba9c4f3b2d80d687b4a56e1d93e260a964e0b375e61de1d27d10c41a43e1f96c"

_BLOB_EVAL = (
    "H4sIAAAAAAAC/61aTW8cyZG9L+D/kCBggIJbrf4k2b0YLLgSZRPQkBpyxoDhNYjsrmQrtdWVPVVdBOmBf8Ue9qyjDjoM5mBg"
    "Lgbcf8zvRWZ9dTU1zbGAGYidlR+RES8jXkTmDwc2Opiqg8jePu/1DzrqYBa773PDtrmO53ns2Jjls7VdObYu8xh/6szdmGSu"
    "I5PoSLrYZJWv2eG7RK11HJtUzd1ylWo1VHP9XmcqMmrt0sTGscuUVv2Tca+nVibDL0ykVZ5o9aAGKnXSA93nehYbdB2Mjsdb"
    "XV1XvdULjTUS1cdExVRd9c9/vMw3H5K1U5FNTOoUpojMXW7iO5P8F0U12cqkFBvSvsVOVH+q3ugsiDnPTbbWCcTe/H8Q8is1"
    "HuPf7v8k0n2A7pAkyFn0H7C/l/QrNZrU+g+n6izG3qEWZTI/l/qd74O+/R7lLzqPKAsFdnE+t5ufEg4JO3we+nKBMOYKmxEB"
    "1K1NdDz1H7wuuNd5atcmtbLXl7leQx+Ou59bl2Di0oY6m0KryqSpS5VNMGZpIuvw552OLVRuYpWaDMaH4rqiRXzIZR7M/cNB"
    "gY8kX2K9uWACPVzKRhHq4G9/+81//FAH3GAfwNnk1qVLWegmstkasgNGuoG5s4TCZzqmvt7pBzUcQG3rPLIaW8HW+icqA1KW"
    "+XuDXQBm/ZE0vHPLGRq6NFAYvrYmMWqkgBaYVWeCSeg4XWv0u7YegdXsKjFzk9m1AD2HNtPEZR3AcO5hmFWtKqt6Q9eJx8Sj"
    "mIRMkQaOUxND2VyKFotVsvl5SWB7REWmudXhoIbTr+W0xnaul1h+BdvGDYHkkPkDxZGE8BDomhztgtbkqBq5jS3IapI8mVvs"
    "AaCZx/mDEekzZdNyA5k6LNTaKXTawTm9s5lAnQJm5t49E4MsXWRiHGMzM8ouEhg9hfRPBt/kqI284T7Ig2+5M2lG4OUJzkBk"
    "sranSzBFwAwd2FyvIBb6UrWD7lgtDc4cVL75eQbBiCA9Q29oiQP7/nvxWZnvcwvB6fT6PMex5egOMMjesH1Y8EE8n192UutX"
    "Q51vUrc6JthWGp4YXjmhDh9F3EvZ8doSLHF9MzpMN5U90TF6NzQYl4vXUHdF1Mgcfpc1aad+yHORGr7vaLcXY3uYt4W0e7sw"
    "ah4kTVVhGuUBBrWLg0u7XgwAyia17gJ2pWdZnv4KR0a52mga7YOm1EQuiYy7oSrM/TrX8Raa4FUQME2mPeTh6ZIMJ4J76ENr"
    "Pj511UseWBxoOTExTxYjrVPLzYd7WA6+ruxaxsJMBojLr/ugR4HwCkcyIgymfunN32Xar9RRdzJSepU6LAU5ljjNpmb5C0D4"
    "Hk7aH4Zb+Gofw8TPiMQdpbPNJ4FF0Igg5R0MoJVOUzvT06afPC5l34WU6usOlxRUjZg3iy3Ygi7XTNUfTl+en6rTq6vz/z4V"
    "rKz4IVVHyvldQldwZA7eCwHnyUg5bsNkvA9MvDNkrDNzqHgbIzhSa/IZdYxz+L82dotU9CcKvt18pOqAIj3XjC8zF2eeefW6"
    "w+PGgIJyVSBBx9Afc7iZHNc9ECJeyXObZJ2KkxK141i6Mr7IxFMRGlASYR6NUno5C7FKnIx4LrhFHoOgE4a5Y7oRzMVtwQ/t"
    "9CKDXtjRNjRelRHHy5wnxdTwxwg03jdPG6SHePj69E+XV4UrpqUievJO4ecykExxTbd5Etn0yZgZ7PAtR/uABsqaM66+N4+z"
    "ci1RCrZwt0A0glPvtzQPPCcYLOnyQweUaOYtCDcEkem4UzODL+EmoaQ+xujIcjdQFU200gtHpJlbnBJ754QhVRO41C5oCypv"
    "0Ktoei1SEW0rUnn0Ers9CrnTEiKygIUGK/GnxfwITr3uSRFcGqydJM8s8kSAWWwby2uZ+HO7n4bJ/OwTmX00eoSC+y+7Ofgb"
    "OathbZy0nF7uDn9dXFKSLF/qpO6N+iNZ9rAgBmSBEsGGMMXm5wQQehZooefu83jzIUPr00ObSN0G4PE+APSeXrxWfIMjs26i"
    "7xToi83D5ic1eDEm6rCB2M5AfCB3nCeekr8Y8lM4S/g/0jPJO5Y6XZusQJYf56nPkGrefCDEapCCBysaw6rlJJ/j2l4OP+AQ"
    "cj4TNuxpyqBap4anbyhj6hLp9Vx6fSVxcruvzwC9CGGFsFtDf4shUxlHf8YguxNWR9W8LVTpAOx6zK2AjZN1dXb97WVHJY4/"
    "xF03UEaIBYmwlyaiCnU9GVFHO9B0spc7Sx09mngZkrsbK0S8mfMBDQjVOLFCr12CzBA5hySCyzwVp9QfqGjzSTd5sbSAR6cw"
    "Hf5M1KScBYyLx53+gZHAZmBT0PLSfQ42fpCr8vsj4ccD2PE4rP/cL1BDDoh2te6UHWH6CYac+BG77B8+bdv+7Zay1PnFH8+u"
    "rk+nJIRZsUZHAlvYfTe40hT+bRFrqcwwLUbMghfX5MhImB/+TQZ00rb+ZB/rL4xjUmT1DQtIooJW2gVkYjtKxE0WeYytLC0z"
    "rl6RcAl/GIRfSCcTYaQmgx/Fv6wRACjvgYLNp0TSWpqWmhgVM9AXSRvPCE8OAxVOvCnQhghfZ9hFphfmyrwLA7OwPGZ2Btri"
    "yUw5/HOw2nxIjS5T/LjcsgTSEmGDUa+1bjPg+Xk4Q7FZzDDiBCMJkY8OH1Z53FTWec7eWHIw2sm1BqPWVNtQfW0Xuew/mHXK"
    "ZIHJnOCA/A6jRWDGg1TSt1g/PeOHLC3s9XtPJFLe5zTj2FnJDXwMg0PffOI0SudMheDXyZLGv4X4+p3jXr2SJoOdFcq5w0cE"
    "uSJ1jcM8jzsceL374J2DJBwL/aYdclmXgE+gB/HR7ZMZy8o1SLzirO+9Ve+L75Lf+f4nvUdYzUnvUVLzttRacD6XUzlhjOPC"
    "km0qocdblLJDRfX4c3wyYAQKsgiBG38hWiNit9HQ3zMOSRX0xou5zamDT1lqKkJLZc1XviG4cDWKP/Q0G142wVmmU8WW7pk4"
    "e8Xy93EP2vguqRcVkX/ld06NugOpExbTPqghUig/s5+lgJOEniz3q/wSj3apTyLrIk9lMVH+UHhMd3BUw005qFh4KqJI/2Mp"
    "BXVH45rvuAaT9fUDzKN+J5/Ra9g97u9CF9t3RDZfhH57efHq7Or0VSAwDMmbj2uioYahITCjDguTqcwuV7F59m/GMBGrBZ7B"
    "nuCZ6Zn1BCazCWgxEvNY/7WdmLFaU9XzxkxaNe8X3osfHCr91zwWAowEAZl9In5SOnUEgRG1ufnoi1/gIx2peckdAnwYseez"
    "Jd0AizRXMnIwDyzSbykBYBmR4DNXJjvHV6txEplDbj1enNTQ9CctIrLWxX0UoXIUNs36Rlkfr1WM4ornMoq9OK4BrlFEmKpD"
    "LCgM/hDdnkmJ4MWY8Wv8or8zgLF9G4HX5xeqNJuUA6r8MXFLyzQ7ZQFqZnUdi4Pxi6OROuTlVDn8C5FqEbOOSCzSvrpziZvb"
    "pWVEaEW42OLo6Jvaog1abRZIK2WrrOMtNY5PtrbrPCT9zFnCSF/YY7knNgvryLK/BwxVFUTrno89V1QstCcl7i1H2MLZ2Zaj"
    "LAaNmh61Nm+H5nmC9N1WXYjXLY0RzDohANN1idAzw9rxGuc0zcx/yvXj1env/QckFCLcg5rbEOciN/cBfR8737gbP3dhlOr+"
    "rNc2+WB/k8eaifmNBrFq2Pqf/0AIE5MthP2C4NoEFC7KYT9FH2TmeXFVCNWFmkvRuVA15+fngmpKRETyQZhldYS0THy9e47O"
    "L69dFxT/+WaHbODOfWmrgmWlYeq6aRsSf868hWO/0TF8h25ZeR7rO5Oh558P/Cps9BuSbPlEGk667uAvLRAM9wYBz8oNPJZd"
    "bmOgHhDkRC2RJaCblPln0B3JTc2KVEJRZ2HquS5vYCLW9Oe5v18urmKlSujPZ8v+bz6zIuYbdnuhKDfGXw8yHUPfL6xaW7GN"
    "BSZ2/lqyDgi5CpdYmjEcpUVw6Eqm3jjhvjxdAAGdPYy+EAKwZbYMO/KPZOWrXFIFZPE7ADDa3wukbu1gQrBZxNqFW4TIUkPC"
    "N3QFhlfsaxiaakCq5ovGkm+ECRpgwCFlPMwb7DWUwD3B3FXu3OXsq/lr64aSTCpnEsveyu2/WpbBnkVnHwN8OlTWDtWhlIEf"
    "VJybhZOq7bPCyEi6fNmDaOJjlozZE3ou5JJByrD+qryKZVkLS2eVpkSlm58WrLYVPl9HuulnTHmlzdwSHdLNBwVIOPqSPGOF"
    "YMv3VM6m0M0XgplXIxsbmmRDqRL+2FYq26hW/++kCUgd3bWYCNpYucOBjlt4NEub6ZtbHW9V9s7ufdlyJRcxgOSEzgAIKx4+"
    "MBYETdQgNFG+PLLdcaqktMUbHz4n8jc+ww6n575YdZO3PkPJwRpD6Q3cruF94GrzMdQIO/7+ddDhrGPl1HELKm+vzr4+vz5V"
    "r0/fXJ/61yaBn4idX15eXZ39/vzKk2fRi6Q6bmYQfkwaM3G81bNUinZE8Psc3usWPysisx8qtoDgVSab5ZdKg3mlheaHun4E"
    "L9uaFf8lCpW/7kMDrTmspipSsoO/CMzcOzuzYsc/H0wekWHymAiTxh4y2/yJ3KH83cbrYF+8WvgzEFG5/LoRhqNJcHYEU7pr"
    "ebQld527gl60df9I5+7Ru4ZL8OtEeipN/jo9sYlQpWLvRG51dOGAqd6einBa+uOueisphiOeXBUYk+LepPmSqw7Uc2Tar88v"
    "zl+eb/7voqteVQyIK7ZERA8BZCnXYa8DATrwcOyw9lkPvuOUP8l57YKpkH2/bomjujChMfz0hqn9KJUZfpTl3FCiCs1bGmq0"
    "ZjCXjWoHYuU89GtTMK/N5PGe97rVb5hgB9oDTOCFxKtuXzn7RtCVRY78RUO5u3qhtQ3t4b7QFkJ0A0KUw6NYCVatinuK+Hhn"
    "WajQatLjS4LNT7VSO4ud9com9LVcOX/NIqEVbIHkiSUqEIK1Tdps8DWfKsldPAm7lMrTIqWehgI8LMrQwOht+eySr0beaUKD"
    "2Zaf2JO2epeOPFLwImWG1z614d52IibP4qTXJo2n315eq/OL6+9e42CcXXx7dh2OBuK+9YkeReYRkfdW4ryLIO9LNMVyv/4M"
    "yMwVdr0mCszbeQ3WVOpcokNoq12oSYt/WyM4fqfnQWjvzIOYbaC28TXaF18gQKlmBhqxZmLbtNNuPpI/QucznBL1EkpQevOj"
    "v4TB4QHziNqUMZNgCU2tdfAEspBkoBbEao6dmCnGy13POidx3MpO4SnhApzSD3nE21mhX0xCYrh3Wb2qNsKqSJQWZsYCW7ik"
    "IpK8a4o2nzJ2aoHn9XdnV6fq1Zl6dfk1HOtlQA45Lri6EfTEm09L60uyOB9B8gYjLQ+C5xpa6AOp9oJJKzLaz8DqURbYVImE"
    "Mzn4O1XFD4V+C1jVdCdcstR66CBB3gsukCCbkkHSu7loY6090DfeF32pi2/4onmRMmFsMs1sDXDwzZ28Sg15jC86GclooAPn"
    "CziIdEsTCGHd+/LK8Jh052To667G01dfuujuSHQanns46fU7TFcdQXvLJ7Th0bcUYyEb0qBw3wO4RCG5m5aL8jG4/1MelR8X"
    "b3iHx0ds6I/6/IFV+GLeV26ZkTn4wsSExzRFVdOnBaHjYCp5kH/HSQLFcstasgO4Ei0V6XYdf9Jr1fGvLt90+TJQqG6MDcAp"
    "8nEClRnym0AfuGJDB0GXaWv/UgyH61qttb9mAvGpJ+6whDym9FqMjNyFmyQzmx/Nr3fBzIt8WlQTRWYTOfmXt5j40kLLj6D5"
    "X0r8Q9R8MQAA"
)

if not RUTA_EVAL.exists():
    RUTA_EVAL.parent.mkdir(parents=True, exist_ok=True)
    RUTA_EVAL.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL)))
    print(f"Eval set escrito en {RUTA_EVAL} (copia embebida).")
else:
    print(f"Se usará el eval set existente en {RUTA_EVAL}.")

eval_set = [json.loads(l) for l in RUTA_EVAL.read_text(encoding="utf-8").splitlines()]

sha_real = hashlib.sha256(RUTA_EVAL.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "… coincide:", sha_real == SHA_EVAL)
print()

from collections import Counter
print(f"Casos: {len(eval_set)}  ->  {dict(Counter(c['bloque'] for c in eval_set))}")
print()
for c in eval_set:
    print(f"  {c['id']:8s} {c['bloque']:13s} {c['subtipo']:26s} {c['input'][:52]}...")

Se usará el eval set existente en data/eval_set_m2.jsonl.
SHA-256: 68c827cccdd90aef … coincide: False

Casos: 21  ->  {'calculo': 12, 'conocimiento': 4, 'adversarial': 5}

  dif-01   calculo       multipaso_encadenado       Un taller compra 3 cajas de tornillos a 18500 pesos ...
  dif-02   calculo       informacion_distractora    En un salón hay 32 estudiantes: 18 son mujeres y 14 ...
  dif-03   calculo       conversion_unidades        Un tanque tiene una capacidad de 2.5 metros cúbicos....
  dif-04   calculo       redondeo_contextual        Una empresa debe transportar 1250 cajas. Cada camión...
  dif-05   calculo       division_decimal           Un lote de 7.5 kilogramos de café se empaca en bolsa...
  dif-06   calculo       porcentaje_encadenado      Una tienda ofrece 20% de descuento y, sobre el preci...
  dif-07   calculo       fraccion_del_resto         Ana leyó 2/5 de un libro el lunes y 1/3 de lo que qu...
  dif-08   calculo       proporcionalidad_inversa   Si 6 obreros constru

In [5]:
# Un caso completo, para ver la estructura.
print(json.dumps(eval_set[5], ensure_ascii=False, indent=2))

{
  "id": "dif-06",
  "bloque": "calculo",
  "subtipo": "porcentaje_encadenado",
  "input": "Una tienda ofrece 20% de descuento y, sobre el precio ya rebajado, un 10% adicional por pago en efectivo. Si el precio original es 200000 pesos, ¿cuánto se paga al final?",
  "esperado": "Paso 1: Aplicamos el primer descuento: 200000 × 0.80 = 160000.\nPaso 2: El segundo descuento se aplica sobre el precio ya rebajado: 160000 × 0.90 = 144000.\nRespuesta final: 144000 pesos",
  "criterio": "Los descuentos sucesivos NO se suman. Responder 140000 (equivalente a un 30% único) es el error clásico.",
  "evaluacion": {
    "tipo": "numerico",
    "valor": "144000"
  }
}


### 3.2 · El eval set de M3: lo que el de M2 no puede medir

`eval_set_m2.jsonl` **no se toca**: es la vara que compara M1, M2, S07, S08 y S10.
Se añaden tres bloques que se reportan **aparte**, en `eval_set_m3_agente.jsonl`
(generado y validado por `scripts/eval_set_m3_fuente.py`):

| Bloque | n | Por qué hace falta |
|---|---|---|
| **D · aritmética** | 13 | El cálculo de M2 está en 10/12 y sus fallos son de *planteamiento*. Aquí hay cuentas donde un 1.5B se equivoca **calculando**: el terreno de las herramientas. |
| **E · compuesto** | 8 | Buscar un dato del colegio **y** calcular con él. Es donde S10 dice que un agente se justifica. |
| **F · mal escrito** | 11 | Versiones con faltas, sin tildes y abreviaturas de casos existentes. Cada una apunta a su `original`: mide **robustez** y **consistencia**. |

Cada caso trae además las **familias de herramientas** que necesita. Las de M2
están en `herramientas_esperadas_m2.jsonl`, un archivo aparte para no modificar
aquel eval set.

In [6]:
import base64, gzip, hashlib, json
from pathlib import Path

_RUTA_EVAL_M3 = Path("data/eval_set_m3_agente.jsonl")
_SHA_EVAL_M3 = "f3078c0f54ba6f0c94c8438fe1585f2f9655adf9c774b474054b45912c5aa2b6"
_BLOB_EVAL_M3 = (
    "H4sIAAAAAAAC/+VcTY/jxtG+51c0BjAwa89IIiVKGr1YBM563hcTrDfOLhIgyOsILbJnlguKLZPiYGaNHHLLOTkEyG2PPuwh"
    "cQADvgRY/RP/kjxVTVJNitKSs+M1HNsLaNTd7K96quqp6qa+PAqDo5k4kkl4OnCOTsTRItJfZCovWy/VOvQllafZYh2uNFUs"
    "swh/RqjwQx3PrxIZB4rahPEqW1OLRzLys0iK4XQ0EZu/Cfds3KMGKl2pRAbcy2cy1cKZiU/L3pY6tZ8QD4XjDKcTx+39f/wU"
    "j2YqXUtxGcYymhVV1KuPiaokNL0mOsj8tRaBEiOx0okYCj+8TGQ6ExH6V0miE5VStfRlkigtUh0LFYlLGUVarDevV6HPj2ex"
    "WOpAoXClsCWbf2izhmsZZbxyjPflUbEpcbbEHHxNTdBCJ1RYTPKPKHyOoeUyVPFapvzkJb5FIX/5vb3Zn6OxXtEAMlJc+zk9"
    "r5PwipaOgjiLIpQEcq3zb3/82Ze2JN1OkoQgmoT4m1iKBTbgSopE+eECGzoVvnwhU+Fjyxx34om1TuIwoo31ZSCxZVLcYu/S"
    "lfSfS+EOy/bLVaSw8J548+9H2eZVvMYj24cxz0BCCrGIijF/vg8vvzZtMZdTGuChcD0zClDCbdwZFQFDPMOHYuhM7Zk2ganW"
    "pA6qTzBFTTMhoSiz/HjzLQSuUwM0A7HO+OBx3zs6hi3RkWZLOce6fLPsCjhKMQrshzPpuyPxEXSyPxzvFdx5JJab13G41ISH"
    "zbdQr823rPusbujhVgzH1N/EtWRpOn8oPKc/cdGEByGxu/1tu+Esr//IlKN+MjT1O8Lmih0Rq1gvUR+wdSABf/roUxFjeddh"
    "AKmonniG7UgEyzFvRlC3Hzt2p/3x4AEtAQaFTY3wo82rFHLvjA0zzbdjwxLQLjYqyOmKk1FLnAThdZiSJ9iDlcIbHHv98QOx"
    "+UYcO17fnT7A/qUhDEN4iT5px7CHMEjYzL3O4hMMFYQJ7fDWCyVs5zFAGF+rJJWzYqS/sUQc7wG5ktGgfzawcPWsHJr8DjY3"
    "UcIZAG+j/lmjw0H5Dmp45ZuvYwLwdvWMn+3SfG7B3XT3IDTqDwkCryUIEhm+nPuZDCAy2WAqIpIZZJTIzeuXomhI+zYZePuN"
    "xi+yNBdQVJhc8hZCRkUfMB6S+5iJ6YhkPiVrQQVNUpyO6jJ8yhNSN9Jfy9ztFwNZpp0HhTlIlViqJTbwpewsS4z9dkmu9FrF"
    "Pr7NsaN+ozw7y3DcVpHh5pc8SpP6ur0hu9VRbyq++/NfhNM7G4/2aupnSch7iAdtvghNmFV6IpLXc6eWYj7O1BU/l5DgZqYe"
    "zp7HQ/uz3tBplC1X1MX7S8ws+SLbvGbh2l78FiAKY5WrJyrL5XeWrBn5ffvxSUuxKt+sY87LjSrCxSZmKrpWtN+mHW3G5IYl"
    "zPRqdAOn6k33C1qmrKDE49abrxK4Q20s4I0gRoieAz2jLk+pLzjyKXnpoSXxIRVPHcuZmwLyFczwJk3ivuGaurwvYn/z9VUc"
    "oh3opFwuaGqYAfh/6ZMLkk+WGi1CeBDIneaahldQcvy9Bh9OV8BJ0hkMmFMLJERXapHI+7bW05aQKK1Mo6Y73puvjYq73/3p"
    "X3slz83Ar4cTz+Zr9AyJZjAZji2RUjOi7FSM6lMHn06TXE1NEwcvZy1MuEI6DKGVtEHECBzW4XV3spWPeS/m+Z0EeNZSgCEo"
    "KdY9Z5ZRj9vEKtl8hR1d5sTaG+A/RLEp9vBKAdOsl9wF2hmmQuYvgo2dfgAHF6cZuMo2VrPaKrGSV6xbEzRMVbrXdV8Uz+Sx"
    "tIIKFnOB8R/0Bg6Z/9HIGdi87LzseWbqqPGEcDaYTrjlbvjGNWaBOwkBnfgkyxelfUfAv0gUx6q+jNdhIIMcTyciYt+zdVhB"
    "dyiZybSCUjGz+0aRM+jK3BvC/2cKirWSCfAuzqbe0M0BhK9cmorwKqPZFtyZGiQpVmBH+UXqoEgP/PwwsydHYgaD6XcojJu6"
    "TqOVoPJmiW9JeSQTgipTu+5sDQO8b5futE3BbYOLpIy6qtLbBlYw6KYJbYkznI777tAZiOeSdjPN8qCc9mkFi0KtegdD+Fc3"
    "VgjPCIJLhY5T1zDI3DlQMRpXoniqhExRSqpctCtLvEa97nt14X766JNcWWdWhJWAaSxVQJ7hmGJB9UItKa0wPhv2HcfzipDc"
    "5PiKlXVX7b73rrFYZ0S4Xbn7/JZc08tGxw6bS14Yu46/Bo4HIvbdn//uDlxvr8h3HqGEmyXY/Hky5J7l7V3qe+RxJNaYhBns"
    "V1tjT8oFlelX4+6LEPKOaj0ZdFPqE/F9hGRO2xwclBJElEx0zhjzJqVoz6kBtK+SS4lsbRbHLlH44YNjw+q9B3uF/XGZh0cP"
    "q0SvQkXeMQjTdRIuMlArxGPuzZtvqEP6PD6FbqFvUzI0BXYi9ubNPyl0G3A1hQCOjZKnKsh8NvrbyCFVS/UCfpnpgnl8Uj64"
    "g6NqizqmfmXCPdqE7fZxIFCMJ07xKJ0HaNpEsBtgDQQSe4hPuDrKgXJwQOENRwcHo0OEPWs8pkhagCrPJ5LXuUt3b/7gVie7"
    "O3335sMPdxr9wcU0T7ePWN8+LNp/WOuVRJ3HcdtuaqWfv0ug0gXz8BY750t0KECy1DuxCURhovQ5lBxstMZLrsjxQIYpYKmW"
    "HNcX4jCRfATvFKmrUJ+AiPjFccMq4w8JBwH8sJLEmnM+ETGXUCOAAE81Qxp8gbYYjQo0R7SjnreXwGzn9ezi/PyEo027K8xQ"
    "UeFo8AHNuGF0S28wkOHIowFlSHrTxoxkb1oHPKfLkmKYWztJ2hPPQp4eCYinl59uUdY0XnMCdktJuycqMZm3oynQPh4mORjD"
    "ej8M+Kiy0bY8WZtp4yXtCOeJfZqbPqph022HTRLZPO98Xpxq1aMvPJeBi8CEiVT6m68hzQHR5hoiTsQQUsa3NVgJBXNUwA0x"
    "aZ+D21HeIJEL+ULTn7AmqSr4NedUweB2YJwaLJIRL5SpphktYTwrZgw4nZQThXfHt3yWLtfUZgjj8kEFzoMSzh/xsvkbk4Rh"
    "UedSXQl8ZhpOb4yyQY9O5+jTfBtPbB/yWBojjakMe5NmHonyuqY8yuQ60RbiU96eEqAmtQAfSLxSF/t4F9LYa3XEV9UMmj15"
    "3eawYoefWGp0WFE4SVdIEToBCWx14kSYs1gWK1UeUphhO4VZ65WaIwbMisTrAWUBy6I0rdsbEY5KU3wiLinhL7a9EJZvS92a"
    "kjr8Ott8ZfQAys7HyWh/RbyFAs+0BDUKI2kE3F0bTkqrbaIgaeZZnxoj0ebIjzQCJkp2p9wKDbBYbMy2P2uu5dMNMN5BMQIy"
    "H3YA5J6zmTI/D8PeUf98aJFqrltvXuU2qERaT9AQDGyeHToAIXouFyrBIwtyJXfCe0e4v5PlLwUiWR60YGwaZfoXa8p0EaOr"
    "y0gXIqpjetQO06SYkgKCOf6HhVjIpBpt6dgCigmuTbs1FiL3EBQCLuJTpiZZRS9oUcwqoRpjj7IvRX/JHYC7Ox/eDDsFRyb5"
    "1Az2EPZ22IhGKt+J4STnCmG31pkRKGlKmu8/PUws6u7nLDxmV1v6DhmaLbzCPbtWh5DXDkJXJK90vookArtEyT0Ml+qJLG5e"
    "oU2FzJoOMAkASZIxfy5v88AZz3GtYRIc1ISooHtRfMEmFcWBkLkHteKTuvyZ26bHjXdnxow48ErHcIrrBLa6BfxqS3jL8NW5"
    "4p9GjH+txfG09+ZbO7B8XJ+KPU88F2y+osBSHDsD+0nwBYewTYnnxsshbr6zTccPBArDEpZhSrn1ClfIEd79uOg9WktsjNln"
    "VkFt7VlcE24o68get0M2XxqyGMkcc3+n+O0L8uwrO4vfGLnRuLF4YWK8eux1K2zi824hnAmsKkTKrZJd4rGcERs3Gs6x90Hz"
    "4dYeJtoZUeN7dcD3SDDrkJq0gxRGYqiwyy2I+KGoi9woqGFK8e6Nn6WSZrIvRuxZYt+ywwOZA2JveWeyKfrq4pPtpRH5epHB"
    "d3LCgQ1akX4wFy4rPvp8f3qBMgMDK32Aj2YHPjps5WiW4rhQNhJjZbrsKugEhhubyTwowya+hXsHBz/64bIIDRteR+y0HWJt"
    "rgkTKONQ30MOS9ItVEAwi+nM0njPauRhWCJlDCCxqEYj4b81H4kVl6MNXNubQ6IYgX7rJAgouxktly/80IFe42letdNGXmn1"
    "Wzmh5Qrrehhxinxp3Q/93qMzfstW6spOWkAEdd7JpaJsDi6IsXf8cRjP12EUEAwXiboO5TpLavc+iLwQdOg+nqSEjrkNlfpZ"
    "zm4CIy+hKWXioN4coQNPt0R7iEBADjd8NSDhMPNS+XwRAwBnM0HHwKCUCXXPNwDAAanv4j4B0MpQePvJBPfGlLGc4azolG3e"
    "lFOmYyqo2stUXWUxk59iZRjdhMz5grYztdY1yzszvZ9x76PRoPkqgqlpPpj+LcJR1hA6TzDSYsUPwkvQqxn+XNBbCQpOJjGH"
    "60zAMakCRD3xmG1zPn9K/vkqxT6n4smvaDXMguy43hnx1I+LRKgxBDAMQ0gR2g0FuKf7ymbl3+u9hyOzU0f7ThlYM9y2msHB"
    "djqHAmqQYrL6Va0AcseCYEG3/KVvUlLLLOHIyHEFHzcbGKfmS0pas9QJlgAuelY+XHDsgK+WmeCBVqkPnLMXGVVjySGhsXml"
    "gU5UMTYinvTU9F9JNFnDzqjh5hsUwOiaJ5pNL1d1Auu0HVg/S8i+GenyNZuLJ789f/rs45nJCuTzPKELR7SFPMNc0xPDxWjE"
    "Nb+vE0Iv12yfnClRjxRkxNcJlervxcx3ShsYZE4PI3N4X8jMbecip3wVHqji5vCIbF8j4aDD5ZJvvAh3MHne+RzrxMq3FsNV"
    "k0JVttNriz2QMni+dtjbTUWZA7BLspzsa6mPRZpnCf+HGd/Tj//PVATo2AR6frg2B2qd4rG5npu+i5R3Cb3R4F0pxg72zLYc"
    "xt6oLfaAAjQKUZ/ThUDXsVdc5bczYyv6l+bOv+SutH+EVdjGKiUWMiS6kHObGuIevzVXmbtrD3/d3oWWdgLcsCXgytipgjq+"
    "H8fpoVSbO+3G/PbYWFcAZy4rF5Cja1uM1Q73DXZgZ10+oGQlSoYn/MFGleNagWXpNjcB7gDK4WFQem1BqQ04krmK5ysZAZnN"
    "FrF8F20JxPGLYj/Kl9H2wdG8vdcSjj+9F9qstxv3Y27c0Qnfzvlt1lBXAZfwexOL8r0J9qJ0Zyd/Z+Ijb/ojeWHiINgm7cD2"
    "X/3ShfWyzX5YTbrE42R3s93T8FTSxjZfFqnfFWl7VUQU/noZ7pJCX1fviOhK4umnfkNkPymg+0LtFOMncsvEukO1X0WmnSlo"
    "k4unANDja+dBaA4pBw38MahdJiaOw86P70EECmIHT5xx0Q2FKCIOY4a1/ZLpTpZxQOeKdBNVfEY5x5RfU9RbjonuTAq2+iZz"
    "K3MbXLdG1cWTT87/9+LJxaOLzV+fkKtfmOvSSfGWanWZFAwl9nutx4MTLOIERJUarE0GCPV8Pf7u11vNwLIcl6p3JpMX5l+N"
    "hK0vpUDyL2lxn8NIPMmLa7tcKU0VvdSjy9EhuTRcRJUu6GgIfDwrbPD2O0TDugCdfB4uQkbU748ccwOeDlN4e6x31WmEHIK5"
    "vGR+4bbeqtWF266+iWFzWPHO7ivvkAcy9GsbFMB54lrTeX6i6Tc+wJeFfJnRtQ7OREpfxsIVC25CjxVJZLqcA2/Ax3NAu46u"
    "VWLum1NWV54IK8TkyG8RmuwRHvyCLxAEfE0c3fPI+/zU44bHM2WNRH1wFzSY17ffQf6dmZ5KJS+g+IGSUb5Wemu9PGY5ETLd"
    "vC76Nklm7nLUn1heqPI7N/wTBVPzEwVo9oDfl+x7FEp4fafxuI7KuyTpnJbG5NnFE8CU3peRLzXztW2evIwFcspmZ5Zdrz8e"
    "iWPiEOXj1Ujhzu/Z8FK/z0DB7M9Bram/Q3eI0dFbAfNErRTZnVqcIMOGXzv4AX/u4CDd99pB5sf/kwnWb1vsR0DrM7Ybugqu"
    "59XfxdqTn+AfuLqh37d6Pz+IdVDeTus8/n//j2pZv4FWgcR/AKo07cIjTQAA"
)
if not _RUTA_EVAL_M3.exists():
    _RUTA_EVAL_M3.parent.mkdir(parents=True, exist_ok=True)
    _RUTA_EVAL_M3.write_bytes(gzip.decompress(base64.b64decode(_BLOB_EVAL_M3)))
    print(f"Escrito {_RUTA_EVAL_M3} (copia embebida).")
eval_m3 = [json.loads(l) for l in _RUTA_EVAL_M3.read_text(encoding="utf-8").splitlines()]
_sha = hashlib.sha256(_RUTA_EVAL_M3.read_bytes()).hexdigest()
print(f"data/eval_set_m3_agente.jsonl: {len(eval_m3)} registros | SHA coincide: {_sha == _SHA_EVAL_M3}")

_RUTA_ANOTACION_M2 = Path("data/herramientas_esperadas_m2.jsonl")
_SHA_ANOTACION_M2 = "da3bd6d94fe22cf5bcc352022bb214a7e75df09740e188f4472e971e6ebf03ef"
_BLOB_ANOTACION_M2 = (
    "H4sIAAAAAAAC/7XUSwqDMBAG4H1PIbNuQaN9XqW4CEmkU9RIYrsR796YlQ+KaRmXwx++DDMkHaCEWwQSi0OcwD6ChzKGV6jq"
    "lluXdFC4qkRf3IEbbCvVouCQu8O6EahrXiqf5n2/68YgowZTajCjBo/U4GkdbLQRQ/R0xFKc3DfXz+t6YbgYvN/xC/UsrsRg"
    "Em853CTgPSnbcon2S79j3YWTVuaXsY02KXQd9DNILV6Vy7RdGbwHGTWYUoMZIcjlO2SG4UvxIKMG0z/ABZJRIBS/6AcykyV2"
    "2wYAAA=="
)
if not _RUTA_ANOTACION_M2.exists():
    _RUTA_ANOTACION_M2.parent.mkdir(parents=True, exist_ok=True)
    _RUTA_ANOTACION_M2.write_bytes(gzip.decompress(base64.b64decode(_BLOB_ANOTACION_M2)))
    print(f"Escrito {_RUTA_ANOTACION_M2} (copia embebida).")
anotacion_m2 = [json.loads(l) for l in _RUTA_ANOTACION_M2.read_text(encoding="utf-8").splitlines()]
_sha = hashlib.sha256(_RUTA_ANOTACION_M2.read_bytes()).hexdigest()
print(f"data/herramientas_esperadas_m2.jsonl: {len(anotacion_m2)} registros | SHA coincide: {_sha == _SHA_ANOTACION_M2}")

from collections import Counter
HERRAMIENTAS_ESPERADAS = {a["id"]: a["herramientas"] for a in anotacion_m2}
HERRAMIENTAS_ESPERADAS.update({c["id"]: c["herramientas"] for c in eval_m3})
print("Bloques M3:", dict(Counter(c["bloque"] for c in eval_m3)))

data/eval_set_m3_agente.jsonl: 32 registros | SHA coincide: True
data/herramientas_esperadas_m2.jsonl: 21 registros | SHA coincide: True
Bloques M3: {'aritmetica': 13, 'compuesto': 8, 'mal_escrito': 11}


### 3.3 · Las respuestas del notebook 3

Cada fila trae la pregunta, la respuesta, los **contextos** que el sistema tuvo
delante y las **observaciones** de las herramientas. Es todo lo que RAGAS necesita,
sin volver a generar nada.

In [7]:
import pandas as pd

RUTA_RESPUESTAS = Path("resultados/respuestas_s10.jsonl")
if not RUTA_RESPUESTAS.exists():
    raise FileNotFoundError("No está resultados/respuestas_s10.jsonl. Ejecuten antes "
                            "S10_Lab_Agente_ReAct_Tutor_Matematicas.ipynb (notebook 3).")

respuestas = [json.loads(l) for l in RUTA_RESPUESTAS.read_text(encoding="utf-8").splitlines()]
detalle = pd.read_csv("resultados/detalle_s10.csv")
ACIERTO = {(r.config, r.id): bool(r.acierto) for r in detalle.itertuples()}
NOMBRES = list(dict.fromkeys(detalle["config"]))

print(f"Respuestas: {len(respuestas)} | configuraciones: {NOMBRES}")
faltan = [(r["config"], r["id"]) for r in respuestas if (r["config"], r["id"]) not in ACIERTO]
assert not faltan, f"hay respuestas sin fila en detalle_s10.csv (¿notebook 3 incompleto?): {faltan[:3]}"

Respuestas: 371 | configuraciones: ['C0 · base', 'C1 · +LoRA', 'C2 · +RAG', 'C3 · +herram.', 'C4 · +ReAct', 'C5 · agente', 'C5-FT · entrenado']


In [8]:
# Referencia y clave de oro de cada caso.
#   referencia : la respuesta esperada del eval set (la usa RAGAS)
#   clave_oro  : el texto que DEBE aparecer en un chunk recuperado (versión objetiva)
# Las claves de conocimiento de M2 son las mismas que midió S08; las de los casos
# compuestos son el campo `dato` del eval set M3; los mal escritos heredan las de su original.
CLAVE_ORO = {
    "con-01": "examen final del periodo equivale al 40",
    "con-02": "octavo se introducen las ecuaciones",
    "con-03": "nota minima aprobatoria es 3.0",
    "con-04": "factor multiplicativo",
}
CLAVE_ORO.update({c["id"]: c["dato"] for c in eval_m3 if c.get("dato")})
for c in eval_m3:
    if c.get("original") in CLAVE_ORO:
        CLAVE_ORO[c["id"]] = CLAVE_ORO[c["original"]]

CASOS = {c["id"]: c for c in eval_set + eval_m3}
ESPERADAS = {a["id"]: a["herramientas"] for a in anotacion_m2}
ESPERADAS.update({c["id"]: c["herramientas"] for c in eval_m3})


def necesita_documentos(cid):
    return "documentos" in ESPERADAS[cid]["familias"]


print("Casos con clave de oro (métricas de contexto):", sorted(CLAVE_ORO))
print("Casos que requieren documentos:", sorted(c for c in CASOS if necesita_documentos(c)))

Casos con clave de oro (métricas de contexto): ['com-01', 'com-02', 'com-03', 'com-04', 'com-05', 'com-06', 'com-07', 'com-08', 'con-01', 'con-02', 'con-03', 'con-04', 'mal-03', 'mal-04', 'mal-07']
Casos que requieren documentos: ['com-01', 'com-02', 'com-03', 'com-04', 'com-05', 'com-06', 'com-07', 'com-08', 'con-01', 'con-02', 'con-03', 'con-04', 'mal-03', 'mal-04', 'mal-07']


---
## 4 · El código de RAGAS (copia literal de `scripts/agente/`)

#### Tool Registry · `registro.py`

El esquema de cada herramienta se **deriva** de su firma y su docstring, en el
formato de tool calling nativo de Qwen2.5. La descripción que ve el modelo y el
código que se ejecuta no pueden divergir. `validar()` convierte lo que propone el
modelo a los tipos reales (acepta `"2,35"`, `"3/4"`, números como texto) y
`ejecutar()` **nunca lanza**: todo error vuelve al modelo como observación.

In [9]:
"""
Tool Registry: herramientas tipadas, con esquema generado y ejecución segura.

Decisión de diseño: el esquema JSON de cada herramienta NO se escribe a mano.
Se deriva de la firma de la función (tipos) y de su docstring (descripciones),
con el mismo formato que espera la plantilla de chat de Qwen2.5 para tool
calling (`{"type": "function", "function": {...}}`). Así la descripción que ve
el modelo y el código que se ejecuta no pueden divergir, y añadir una
herramienta es escribir UNA función decorada: el núcleo del agente no cambia.

El registro separa tres responsabilidades que el agente nunca mezcla:
  esquemas()  -> lo que se le MUESTRA al modelo
  validar()   -> convertir los argumentos que PROPONE el modelo a los tipos reales
  ejecutar()  -> correr la función y devolver un resultado tipado (nunca lanza)
"""

from __future__ import annotations

import inspect
import json
import re
import time
import typing
from dataclasses import asdict, dataclass, field
from fractions import Fraction
from typing import Any, Callable, Literal, get_args, get_origin


class ErrorHerramienta(Exception):
    """Error esperado de dominio (p. ej. división entre cero). Su mensaje se
    devuelve al modelo como observación, para que lo explique o corrija."""


# --------------------------------------------------------------------------
# Tipos de datos del protocolo agente <-> herramientas
# --------------------------------------------------------------------------

@dataclass
class LlamadaHerramienta:
    nombre: str
    argumentos: dict
    formato: str = "nativo"        # nativo (<tool_call>) | recuperado (JSON suelto)


@dataclass
class ResultadoHerramienta:
    herramienta: str
    argumentos: dict
    ok: bool
    resultado: str | None = None   # texto legible para el modelo
    valor: str | None = None       # forma canónica ("73/72", "9.316") para métricas
    error: str | None = None
    tipo_error: str | None = None  # desconocida | argumentos | dominio | interno
    ms: float = 0.0
    extra: dict = field(default_factory=dict)

    def como_observacion(self) -> str:
        if self.ok:
            d = {"ok": True, "resultado": self.resultado}
            if self.valor is not None and self.valor != self.resultado:
                d["valor"] = self.valor
        else:
            d = {"ok": False, "error": self.error}
        return json.dumps(d, ensure_ascii=False)

    def a_dict(self) -> dict:
        return asdict(self)


@dataclass
class Herramienta:
    nombre: str
    descripcion: str
    familia: str
    funcion: Callable
    parametros: dict
    requeridos: list
    comodin_de: tuple = ()          # familias que esta herramienta también puede cubrir

    def esquema(self) -> dict:
        return {"type": "function", "function": {
            "name": self.nombre,
            "description": self.descripcion,
            "parameters": {"type": "object", "properties": self.parametros,
                           "required": self.requeridos},
        }}


# --------------------------------------------------------------------------
# Esquema a partir de tipos y docstring
# --------------------------------------------------------------------------

_TIPOS_JSON = {int: "integer", float: "number", str: "string", bool: "boolean"}


def _esquema_tipo(anotacion) -> dict:
    origen = get_origin(anotacion)
    if origen is Literal:
        return {"type": "string", "enum": list(get_args(anotacion))}
    if origen in (list, typing.List):
        (interno,) = get_args(anotacion) or (float,)
        return {"type": "array", "items": _esquema_tipo(interno)}
    if anotacion in _TIPOS_JSON:
        return {"type": _TIPOS_JSON[anotacion]}
    raise TypeError(f"tipo no soportado en una herramienta: {anotacion!r}")


def _parsear_docstring(doc: str) -> tuple[str, dict]:
    """Formato Google: descripción, y luego una sección 'Args:' con 'nombre: texto'."""
    doc = inspect.cleandoc(doc or "")
    partes = re.split(r"^\s*Args:\s*$", doc, maxsplit=1, flags=re.MULTILINE)
    descripcion = " ".join(partes[0].split())
    args: dict = {}
    if len(partes) > 1:
        actual = None
        for linea in partes[1].splitlines():
            m = re.match(r"^\s*(\w+)\s*:\s*(.*)$", linea)
            if m:
                actual = m.group(1)
                args[actual] = m.group(2).strip()
            elif actual and linea.strip():
                args[actual] += " " + linea.strip()
    return descripcion, args


# --------------------------------------------------------------------------
# Conversión de argumentos: lo que escribe un LLM pequeño no siempre es JSON limpio
# --------------------------------------------------------------------------

def a_fraccion(valor: Any) -> Fraction:
    """Número exacto a partir de lo que proponga el modelo: 12, 2.35, "2,35",
    "18.500"?, "3/4", "1 1/2"... Se trabaja con Fraction para no introducir
    errores de coma flotante en una herramienta cuyo propósito es no equivocarse."""
    if isinstance(valor, bool):
        raise ValueError("un booleano no es un número")
    if isinstance(valor, int):
        return Fraction(valor)
    if isinstance(valor, float):
        return Fraction(repr(valor))
    if isinstance(valor, Fraction):
        return valor
    if not isinstance(valor, str):
        raise ValueError(f"no es un número: {valor!r}")
    t = valor.strip().replace("−", "-").replace(" ", "").replace("%", "")
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+(\.\d+)?", t):      # 18,500 -> miles
        t = t.replace(",", "")
    elif re.fullmatch(r"-?\d+,\d+", t):                     # 2,35 -> decimal
        t = t.replace(",", ".")
    m = re.fullmatch(r"(-?\d+)\s*\+?\s*(\d+)/(\d+)", valor.strip())  # "1 1/2"
    if m and " " in valor.strip():
        entero, num, den = map(int, m.groups())
        signo = -1 if entero < 0 else 1
        return Fraction(entero) + signo * Fraction(num, den)
    try:
        return Fraction(t)
    except (ValueError, ZeroDivisionError) as e:
        raise ValueError(f"no es un número: {valor!r}") from e


def _convertir(valor: Any, anotacion, nombre: str):
    origen = get_origin(anotacion)
    if origen is Literal:
        opciones = get_args(anotacion)
        v = str(valor).strip().lower()
        if v not in opciones:
            raise ValueError(f"'{nombre}' debe ser uno de {list(opciones)}, no {valor!r}")
        return v
    if origen in (list, typing.List):
        if isinstance(valor, str):
            try:
                valor = json.loads(valor)
            except json.JSONDecodeError:
                valor = [p for p in re.split(r"[;\s]+|,(?=\s)", valor.strip("[] ")) if p]
        if not isinstance(valor, (list, tuple)) or not valor:
            raise ValueError(f"'{nombre}' debe ser una lista no vacía")
        (interno,) = get_args(anotacion) or (float,)
        return [_convertir(v, interno, nombre) for v in valor]
    if anotacion is float:
        return a_fraccion(valor)
    if anotacion is int:
        f = a_fraccion(valor)
        if f.denominator != 1:
            raise ValueError(f"'{nombre}' debe ser entero, no {valor!r}")
        return int(f)
    if anotacion is str:
        if isinstance(valor, (dict, list)):
            raise ValueError(f"'{nombre}' debe ser texto")
        return str(valor)
    if anotacion is bool:
        return bool(valor)
    return valor


# --------------------------------------------------------------------------
# El registro
# --------------------------------------------------------------------------

class RegistroHerramientas:
    def __init__(self):
        self._herramientas: dict[str, Herramienta] = {}

    # ---- alta de herramientas -------------------------------------------
    def herramienta(self, familia: str, comodin_de: tuple = (), nombre: str | None = None):
        """Decorador: `@registro.herramienta(familia="aritmetica")`."""
        def decorar(fn: Callable) -> Callable:
            self.registrar(fn, familia=familia, comodin_de=comodin_de, nombre=nombre)
            return fn
        return decorar

    def registrar(self, fn: Callable, familia: str, comodin_de: tuple = (),
                  nombre: str | None = None) -> Herramienta:
        nombre = nombre or fn.__name__
        if nombre in self._herramientas:
            raise ValueError(f"ya existe una herramienta llamada {nombre!r}")
        descripcion, docs_args = _parsear_docstring(fn.__doc__)
        if not descripcion:
            raise ValueError(f"la herramienta {nombre!r} necesita una descripción (docstring)")
        hints = typing.get_type_hints(fn, include_extras=False)
        parametros, requeridos = {}, []
        for p in inspect.signature(fn).parameters.values():
            if p.name not in hints:
                raise TypeError(f"{nombre}: el parámetro {p.name!r} no tiene tipo")
            esquema = _esquema_tipo(hints[p.name])
            if p.name not in docs_args:
                raise ValueError(f"{nombre}: falta describir {p.name!r} en 'Args:'")
            esquema["description"] = docs_args[p.name]
            if p.default is inspect.Parameter.empty:
                requeridos.append(p.name)
            else:
                esquema["default"] = p.default
            parametros[p.name] = esquema
        h = Herramienta(nombre, descripcion, familia, fn, parametros, requeridos, tuple(comodin_de))
        self._herramientas[nombre] = h
        return h

    # ---- consulta ----------------------------------------------------------
    def __contains__(self, nombre: str) -> bool:
        return nombre in self._herramientas

    def __len__(self) -> int:
        return len(self._herramientas)

    def obtener(self, nombre: str) -> Herramienta:
        return self._herramientas[nombre]

    def nombres(self) -> list[str]:
        return list(self._herramientas)

    def esquemas(self) -> list[dict]:
        return [h.esquema() for h in self._herramientas.values()]

    def familia_de(self, nombre: str) -> str | None:
        h = self._herramientas.get(nombre)
        return h.familia if h else None

    def subconjunto(self, incluir=None, excluir=()) -> "RegistroHerramientas":
        """Vista con parte de las herramientas (p. ej. C4 sin `buscar_documentos`)."""
        nuevo = RegistroHerramientas()
        for n, h in self._herramientas.items():
            if (incluir is None or n in incluir) and n not in excluir:
                nuevo._herramientas[n] = h
        return nuevo

    # ---- validación y ejecución -------------------------------------------
    def validar(self, llamada: LlamadaHerramienta) -> dict:
        """Devuelve los argumentos convertidos o lanza ValueError con un mensaje
        pensado para que el MODELO lo lea y corrija la llamada."""
        h = self._herramientas[llamada.nombre]
        hints = typing.get_type_hints(h.funcion)
        args = llamada.argumentos if isinstance(llamada.argumentos, dict) else {}
        desconocidos = set(args) - set(h.parametros)
        if desconocidos:
            raise ValueError(f"argumentos desconocidos para {h.nombre}: {sorted(desconocidos)}; "
                             f"acepta {list(h.parametros)}")
        faltan = [r for r in h.requeridos if r not in args]
        if faltan:
            raise ValueError(f"faltan argumentos para {h.nombre}: {faltan}")
        return {k: _convertir(v, hints[k], k) for k, v in args.items()}

    def ejecutar(self, llamada: LlamadaHerramienta) -> ResultadoHerramienta:
        t0 = time.perf_counter()
        base = dict(herramienta=llamada.nombre, argumentos=llamada.argumentos)

        def fin(**kw):
            return ResultadoHerramienta(**base, **kw, ms=(time.perf_counter() - t0) * 1000)

        if llamada.nombre not in self._herramientas:
            return fin(ok=False, tipo_error="desconocida",
                       error=f"no existe la herramienta {llamada.nombre!r}. Disponibles: {self.nombres()}")
        try:
            args = self.validar(llamada)
        except (ValueError, TypeError) as e:
            return fin(ok=False, tipo_error="argumentos", error=str(e))
        try:
            salida = self._herramientas[llamada.nombre].funcion(**args)
        except ErrorHerramienta as e:
            return fin(ok=False, tipo_error="dominio", error=str(e))
        except Exception as e:  # noqa: BLE001 — una herramienta nunca tumba al agente
            return fin(ok=False, tipo_error="interno", error=f"{type(e).__name__}: {e}")

        if isinstance(salida, dict):
            return fin(ok=True, resultado=str(salida.get("resultado")), valor=salida.get("valor"),
                       extra={k: v for k, v in salida.items() if k not in ("resultado", "valor")})
        return fin(ok=True, resultado=str(salida), valor=str(salida))

#### Núcleo del agente · `nucleo.py`

- **Normalizador**: reglas deterministas + reescritura LLM opcional, con la
  salvaguarda de que los números no cambien.
- **Parser**: `<tool_call>` nativo, JSON suelto recuperado o error de formato
  devuelto al modelo.
- **Bucle ReAct** (`AgenteTutor`): una sola clase configura C2–C5 con
  `contexto_fijo`, `max_rondas`, `verificar`, `normalizador` y `few_shot`.
- **Verificador**: si la respuesta no usa ningún resultado de las herramientas,
  se pide una revisión.

In [10]:
"""
Núcleo del agente: normalizador de consulta, parser de acciones y bucle ReAct.

El núcleo no conoce ninguna herramienta concreta ni ningún modelo concreto:

  llm(mensajes, esquemas) -> texto      cualquier función (Qwen, un simulador de tests)
  RegistroHerramientas                  qué herramientas hay y cómo se ejecutan

Flujo de una pregunta (la arquitectura de la propuesta M3):

  Usuario
    -> Normalizador        (reglas deterministas + reescritura LLM opcional, con salvaguarda de números)
    -> Razonador (LLM)     pensamiento + propuesta de acción, en el formato nativo <tool_call> de Qwen2.5
    -> Parser/Validador    texto -> LlamadaHerramienta (o error de formato que se devuelve al modelo)
    -> Tool Registry       validación de tipos + ejecución segura -> ResultadoHerramienta
    -> Observación         vuelve al modelo como <tool_response>
    -> ... repetir hasta que el modelo responda o se agote el tope de pasos
    -> Verificador         ¿la respuesta final usa algún resultado de las herramientas?
    -> Respuesta final + traza completa (para métricas y RAGAS)
"""

from __future__ import annotations

import json
import re
import time
from dataclasses import asdict, dataclass, field
from typing import Callable



# --------------------------------------------------------------------------
# Utilidades numéricas (independientes de CODIGO_METRICAS)
# --------------------------------------------------------------------------

_PAT_NUM = re.compile(r"-?\d+(?:[.,]\d+)?(?:\s*/\s*\d+)?")


def numeros_en(texto: str) -> list[str]:
    """Números de un texto con punto decimal. Como en todo el proyecto, el punto
    es SIEMPRE decimal (9.316) y la coma solo es de miles si le siguen 3 cifras."""
    t, previo = texto or "", None
    while t != previo:                      # 1,138,712 necesita dos pasadas
        previo, t = t, re.sub(r"(\d),(\d{3})(?!\d)", r"\1\2", t)
    return [re.sub(r"\s+", "", n).replace(",", ".") for n in _PAT_NUM.findall(t)]


def a_numero(texto: str | None) -> float | None:
    if texto is None:
        return None
    try:
        if "/" in texto:
            a, b = texto.split("/")
            return float(a) / float(b)
        return float(texto)
    except (ValueError, ZeroDivisionError):
        return None


def mismo_numero(a, b, tol: float = 1e-6) -> bool:
    x, y = a_numero(str(a)) if a is not None else None, a_numero(str(b)) if b is not None else None
    return x is not None and y is not None and abs(x - y) <= tol * max(1.0, abs(y))


_PAT_RESPUESTA = re.compile(r"Respuesta\s+final\s*:\s*(.+)", re.IGNORECASE)


def valor_final(texto: str) -> str | None:
    m = _PAT_RESPUESTA.search(texto or "")
    fuente = m.group(1).splitlines()[0] if m else (texto or "")
    nums = numeros_en(fuente)
    if not nums:
        return None
    return nums[0] if m else nums[-1]


# --------------------------------------------------------------------------
# Normalizador de la consulta (preguntas mal escritas)
# --------------------------------------------------------------------------
# Decisión: dos capas.
#  1) REGLAS deterministas y auditables para lo frecuente y seguro: espacios,
#     signos repetidos, abreviaturas de chat y operadores escritos en palabras
#     ENTRE números ("17/24 mas 11/36", "3847 x 296").
#  2) REESCRITURA con el LLM (opcional, solo en la arquitectura agéntica) para
#     faltas de ortografía que ninguna regla cubre ("bale", "evaluasion").
# Salvaguarda común: si la versión corregida no conserva EXACTAMENTE los mismos
# números, se descarta. Una corrección ortográfica que cambia un dato es peor
# que la pregunta mal escrita.

_ABREVIATURAS = [
    (r"\bq\b", "que"), (r"\bxq\b", "porque"), (r"\bpq\b", "porque"), (r"\bpa\b", "para"),
    # "x" solo significa "por" delante de palabras típicas; "halla x si..." debe quedar igual.
    (r"\bx\b(?=\s+(?:pagar|persona|personas|cada|dia|día|mes|periodo|período|ciento|favor|hora|kilo)\b)", "por"),
    (r"\bai\b", "hay"), (r"\bk\b", "que"),
    (r"\btmb\b", "también"), (r"\bdl\b", "del"),
]
_OPERADORES_ENTRE_NUMEROS = [
    (r"(?<=\d)\s+(?:mas|más)\s+(?=\d)", " + "),
    (r"(?<=\d)\s+menos\s+(?=\d)", " - "),
    (r"(?<=\d)\s*[x×*]\s*(?=\d)", " × "),
    (r"(?<=\d)\s+por\s+(?=\d)", " × "),
    (r"(?<=\d)\s+(?:dividido\s+(?:en|entre|por)|entre)\s+(?=\d)", " ÷ "),
]


@dataclass
class ConsultaNormalizada:
    original: str
    normalizada: str
    cambios: list = field(default_factory=list)
    reescrita_llm: bool = False
    descartada: str | None = None      # motivo si la reescritura LLM se rechazó


def parece_informal(texto: str) -> bool:
    """Disparador de la reescritura LLM: sin tildes, sin eñes y sin '¿'. Una
    pregunta bien escrita en español casi siempre tiene alguno de los tres; así
    no se gasta una generación (ni se arriesga el sentido) en las que no lo necesitan."""
    return not re.search(r"[¿áéíóúñÁÉÍÓÚÑ]", texto)


class Normalizador:
    def __init__(self, reescribir: Callable[[str], str] | None = None,
                 condicion: Callable[[str], bool] | None = parece_informal):
        self.reescribir = reescribir
        self.condicion = condicion

    @staticmethod
    def _mismos_numeros(a: str, b: str) -> bool:
        return sorted(numeros_en(a)) == sorted(numeros_en(b))

    def reglas(self, texto: str) -> tuple[str, list]:
        cambios, t = [], " ".join(texto.split())
        for patron, reemplazo in _OPERADORES_ENTRE_NUMEROS + _ABREVIATURAS:
            nuevo = re.sub(patron, reemplazo, t, flags=re.IGNORECASE)
            if nuevo != t:
                cambios.append(patron)
                t = nuevo
        t2 = re.sub(r"([?!.])\1+", r"\1", t)
        if t2 != t:
            cambios.append("signos repetidos")
        t = t2
        if "?" in t and "¿" not in t:
            t = "¿" + t[0].upper() + t[1:] if t else t
        elif t:
            t = t[0].upper() + t[1:]
        return t, cambios

    def __call__(self, texto: str) -> ConsultaNormalizada:
        t, cambios = self.reglas(texto)
        if not self._mismos_numeros(texto, t):     # las reglas nunca deben tocar números
            return ConsultaNormalizada(texto, texto, [], descartada="las reglas alteraban números")
        res = ConsultaNormalizada(texto, t, cambios)
        if self.reescribir is not None and (self.condicion is None or self.condicion(texto)):
            candidata = (self.reescribir(t) or "").strip().strip('"').splitlines()
            candidata = candidata[0].strip() if candidata else ""
            if not candidata:
                res.descartada = "reescritura vacía"
            elif not self._mismos_numeros(t, candidata):
                res.descartada = f"la reescritura cambió los números: {candidata!r}"
            elif not (0.5 <= len(candidata) / max(1, len(t)) <= 2.0):
                res.descartada = "la reescritura cambió demasiado la longitud"
            else:
                res.normalizada, res.reescrita_llm = candidata, candidata != t
        return res


# --------------------------------------------------------------------------
# Parser: texto del modelo -> pensamiento + llamadas | respuesta
# --------------------------------------------------------------------------

@dataclass
class SalidaModelo:
    texto: str
    pensamiento: str = ""
    llamadas: list = field(default_factory=list)
    respuesta: str | None = None
    error_formato: str | None = None


_PAT_TOOL_CALL = re.compile(r"<tool_call>\s*(.*?)\s*(?:</tool_call>|$)", re.DOTALL)


def _json_de(texto: str):
    """Primer objeto JSON balanceado del texto (tolera texto alrededor)."""
    inicio = texto.find("{")
    while inicio != -1:
        nivel, en_cadena, escape = 0, False, False
        for i in range(inicio, len(texto)):
            c = texto[i]
            if en_cadena:
                if escape:
                    escape = False
                elif c == "\\":
                    escape = True
                elif c == '"':
                    en_cadena = False
                continue
            if c == '"':
                en_cadena = True
            elif c == "{":
                nivel += 1
            elif c == "}":
                nivel -= 1
                if nivel == 0:
                    try:
                        return json.loads(texto[inicio:i + 1])
                    except json.JSONDecodeError:
                        break
        inicio = texto.find("{", inicio + 1)
    return None


def _a_llamada(obj, formato: str) -> LlamadaHerramienta | None:
    if not isinstance(obj, dict):
        return None
    nombre = obj.get("name") or obj.get("tool") or obj.get("nombre")
    args = obj.get("arguments", obj.get("args", obj.get("argumentos", {})))
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}
    if not isinstance(nombre, str):
        return None
    return LlamadaHerramienta(nombre=nombre, argumentos=args if isinstance(args, dict) else {},
                              formato=formato)


def analizar_salida(texto: str) -> SalidaModelo:
    s = SalidaModelo(texto=texto)
    bloques = _PAT_TOOL_CALL.findall(texto)
    if "<tool_call>" in texto:
        s.pensamiento = texto.split("<tool_call>")[0].strip()
        for b in bloques:
            llamada = _a_llamada(_json_de(b), "nativo")
            if llamada is None:
                s.error_formato = f"no pude leer la llamada a herramienta: {b[:120]!r}"
            else:
                s.llamadas.append(llamada)
        if not s.llamadas and s.error_formato is None:
            s.error_formato = "etiqueta <tool_call> vacía"
    else:
        # Recuperación: el modelo escribió el JSON sin las etiquetas.
        obj = _json_de(texto)
        llamada = _a_llamada(obj, "recuperado") if obj else None
        if llamada is not None and ("arguments" in obj or "args" in obj):
            s.pensamiento = texto[:texto.find("{")].strip()
            s.llamadas.append(llamada)
        else:
            s.respuesta = texto.strip()
    s.pensamiento = re.sub(r"^\s*(Pensamiento|Thought)\s*:\s*", "", s.pensamiento, flags=re.IGNORECASE)
    return s


# --------------------------------------------------------------------------
# Traza
# --------------------------------------------------------------------------

@dataclass
class EventoTraza:
    paso: int
    tipo: str        # normalizacion | contexto | pensamiento | llamada | observacion |
                     # error_formato | verificacion | respuesta | tope_pasos
    datos: dict


@dataclass
class RespuestaAgente:
    pregunta: str
    respuesta: str
    pregunta_normalizada: str
    traza: list = field(default_factory=list)
    contextos: list = field(default_factory=list)       # chunks usados (fijos o buscados)
    observaciones: list = field(default_factory=list)   # ResultadoHerramienta como dict
    pasos_llm: int = 0
    terminacion: str = "respuesta"                      # respuesta | tope_pasos
    segundos: float = 0.0

    @property
    def llamadas(self) -> list:
        return self.observaciones

    def a_dict(self) -> dict:
        d = asdict(self)
        d["traza"] = [asdict(e) if not isinstance(e, dict) else e for e in self.traza]
        return d


# --------------------------------------------------------------------------
# El agente
# --------------------------------------------------------------------------

class AgenteTutor:
    """Una sola clase configurable para las configuraciones C2–C5.

    contexto_fijo : función pregunta -> [chunks] que se INYECTA siempre (RAG clásico).
                    None = el agente decide si buscar (vía herramienta `buscar_documentos`).
    registro      : herramientas disponibles; None = sin herramientas.
    max_rondas    : cuántas veces puede pedir herramientas (1 = function calling de una
                    ronda, C3; >1 = ReAct, C4/C5).
    verificar     : si la respuesta final no coincide con ningún resultado numérico
                    observado, se le pide UNA revisión.
    normalizador  : Normalizador o None.
    few_shot      : lista de conversaciones de ejemplo (mensajes) que van antes de la pregunta.
    """

    def __init__(self, llm: Callable, system: str, registro: RegistroHerramientas | None = None,
                 contexto_fijo: Callable | None = None, max_rondas: int = 1, max_pasos: int = 6,
                 verificar: bool = False, normalizador: Normalizador | None = None,
                 few_shot: list | None = None, nombre: str = "agente",
                 extraer_contextos: Callable | None = None):
        self.llm, self.system, self.registro = llm, system, registro
        self.contexto_fijo, self.max_rondas, self.max_pasos = contexto_fijo, max_rondas, max_pasos
        self.verificar, self.normalizador, self.few_shot = verificar, normalizador, few_shot or []
        self.nombre = nombre
        # Cómo sacar chunks de una observación de `buscar_documentos` (para RAGAS).
        self.extraer_contextos = extraer_contextos

    # ------------------------------------------------------------------
    def _mensajes_iniciales(self, pregunta: str, contextos: list) -> list:
        msgs = [{"role": "system", "content": self.system}]
        for conversacion in self.few_shot:
            msgs.extend(conversacion)
        if contextos:
            bloque = "\n\n".join(contextos)
            msgs.append({"role": "user", "content": f"Contexto:\n{bloque}\n\nPregunta: {pregunta}"})
        else:
            msgs.append({"role": "user", "content": pregunta})
        return msgs

    @staticmethod
    def _msg_llamadas(pensamiento: str, llamadas: list) -> dict:
        return {"role": "assistant", "content": pensamiento,
                "tool_calls": [{"type": "function",
                                "function": {"name": l.nombre, "arguments": l.argumentos}}
                               for l in llamadas]}

    # ------------------------------------------------------------------
    def __call__(self, pregunta: str) -> RespuestaAgente:
        t0 = time.perf_counter()
        traza: list = []
        paso = 0

        norm = self.normalizador(pregunta) if self.normalizador else None
        q = norm.normalizada if norm else pregunta
        if norm:
            traza.append(EventoTraza(0, "normalizacion", asdict(norm)))

        contextos = list(self.contexto_fijo(q)) if self.contexto_fijo else []
        if contextos:
            traza.append(EventoTraza(0, "contexto", {"chunks": contextos}))

        msgs = self._mensajes_iniciales(q, contextos)
        esquemas = self.registro.esquemas() if self.registro is not None and len(self.registro) else None
        observaciones: list = []
        rondas, revisado, pasos_llm = 0, False, 0

        while pasos_llm < self.max_pasos:
            paso += 1
            herramientas_activas = esquemas if (esquemas and rondas < self.max_rondas) else None
            texto = self.llm(msgs, herramientas_activas)
            pasos_llm += 1
            salida = analizar_salida(texto)

            if salida.llamadas and herramientas_activas:
                if salida.pensamiento:
                    traza.append(EventoTraza(paso, "pensamiento", {"texto": salida.pensamiento}))
                msgs.append(self._msg_llamadas(salida.pensamiento, salida.llamadas))
                for llamada in salida.llamadas:
                    traza.append(EventoTraza(paso, "llamada", asdict(llamada)))
                    res = self.registro.ejecutar(llamada)
                    d = res.a_dict()
                    d["formato"] = llamada.formato
                    observaciones.append(d)
                    traza.append(EventoTraza(paso, "observacion", d))
                    if res.ok and self.extraer_contextos and llamada.nombre == "buscar_documentos":
                        contextos.extend(self.extraer_contextos(res))
                    msgs.append({"role": "tool", "name": llamada.nombre,
                                 "content": res.como_observacion()})
                rondas += 1
                continue

            if salida.llamadas and not herramientas_activas:
                # Pidió herramientas cuando ya no le quedan rondas: se le pide responder.
                traza.append(EventoTraza(paso, "error_formato",
                                         {"motivo": "llamada sin rondas disponibles", "texto": texto}))
                msgs.append({"role": "assistant", "content": texto})
                msgs.append({"role": "user", "content": "Ya no puedes usar más herramientas. "
                             "Responde con lo que tienes y termina con 'Respuesta final:'."})
                continue

            if salida.error_formato:
                traza.append(EventoTraza(paso, "error_formato", {"motivo": salida.error_formato, "texto": texto}))
                msgs.append({"role": "assistant", "content": texto})
                msgs.append({"role": "user", "content": f"Error: {salida.error_formato}. Escribe la llamada "
                             "como JSON válido dentro de <tool_call></tool_call> o responde directamente."})
                continue

            respuesta = salida.respuesta or ""
            if self.verificar and not revisado:
                problema = self._verificar(respuesta, observaciones)
                if problema:
                    revisado = True
                    traza.append(EventoTraza(paso, "verificacion", {"problema": problema, "respuesta": respuesta}))
                    msgs.append({"role": "assistant", "content": respuesta})
                    msgs.append({"role": "user", "content": f"Verificación: {problema} Revisa tu respuesta "
                                 "usando los resultados de las herramientas y vuelve a escribirla completa, "
                                 "terminando con 'Respuesta final:'."})
                    continue
            traza.append(EventoTraza(paso, "respuesta", {"texto": respuesta}))
            return RespuestaAgente(pregunta, respuesta, q, traza, contextos, observaciones,
                                   pasos_llm, "respuesta", time.perf_counter() - t0)

        # Tope de pasos: una última generación SIN herramientas.
        traza.append(EventoTraza(paso, "tope_pasos", {"max_pasos": self.max_pasos}))
        msgs.append({"role": "user", "content": "Se acabaron los pasos. Responde ya con lo que tienes "
                     "y termina con 'Respuesta final:'."})
        respuesta = analizar_salida(self.llm(msgs, None)).respuesta or ""
        pasos_llm += 1
        traza.append(EventoTraza(paso + 1, "respuesta", {"texto": respuesta}))
        return RespuestaAgente(pregunta, respuesta, q, traza, contextos, observaciones,
                               pasos_llm, "tope_pasos", time.perf_counter() - t0)

    # ------------------------------------------------------------------
    @staticmethod
    def _verificar(respuesta: str, observaciones: list) -> str | None:
        """Consistencia respuesta <-> herramientas. Solo interviene si hubo
        resultados numéricos y la respuesta final no usa NINGUNO de ellos."""
        valores = [o.get("valor") for o in observaciones if o.get("ok") and o.get("valor")]
        valores = [v for v in valores if a_numero(v) is not None]
        if not valores:
            return None
        v = valor_final(respuesta)
        if v is None:
            return "tu respuesta no tiene un resultado numérico final."
        if any(mismo_numero(v, x) for x in valores):
            return None
        return (f"tu respuesta final ({v}) no coincide con ningún resultado obtenido con las "
                f"herramientas ({', '.join(valores[-3:])}).")

#### RAGAS · `ragas_tutor.py`

Las cuatro métricas de S10 con un evaluador LLM, y una versión objetiva de cada
una para calibrarlo.

In [11]:
"""
RAGAS adaptado al tutor: las cuatro métricas de S10, calculadas "por dentro"
con un LLM evaluador local, más una versión OBJETIVA de cada una que sirve para
calibrarlo (como se calibró el juez en M2 contra la verdad numérica).

Qué mide cada una, en este dominio:

  faithfulness       ¿cada afirmación de la respuesta se apoya en lo que el sistema
                     OBSERVÓ? Para un agente, lo observado son los documentos
                     recuperados Y los resultados de las herramientas. Un cálculo
                     escrito "de cabeza", sin herramienta ni documento que lo
                     respalde, no es fiel aunque sea correcto: es la alerta de S10
                     ("acertó por la razón equivocada").
  context_precision  De los chunks recuperados, ¿los útiles quedaron arriba?
                     (average precision sobre el ranking)
  context_recall     ¿Lo recuperado contiene todo lo necesario para la referencia?
  answer_relevancy   ¿La respuesta va al grano? Se generan preguntas a partir de la
                     respuesta y se compara su similitud con la pregunta real.

Versiones objetivas (sin LLM):
  faithfulness_numerica   fracción de números de la respuesta que aparecen en la
                          pregunta, los documentos o las herramientas.
  context_precision_oro   relevancia = el chunk contiene la clave de oro.
  context_recall_oro      la clave de oro está en algún chunk recuperado.

El evaluador es cualquier función `llm(system, user, max_new_tokens) -> texto`.
"""

from __future__ import annotations

import re
import unicodedata



def _norm(texto: str) -> str:
    t = unicodedata.normalize("NFKD", (texto or "").lower())
    return "".join(c for c in t if not unicodedata.combining(c))


# --------------------------------------------------------------------------
# Utilidades
# --------------------------------------------------------------------------

def afirmaciones(respuesta: str) -> list[str]:
    """Descompone la respuesta en afirmaciones verificables (una por paso/oración).
    Se quita el rótulo 'Paso N:' para que su número no cuente como dato."""
    texto = re.sub(r"Paso\s*\d+\s*:", "\n", respuesta or "", flags=re.IGNORECASE)
    partes = re.split(r"\n+|(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¿])", texto)
    return [p.strip(" -•") for p in partes if len(p.strip().split()) >= 3]


def veredictos(texto: str, n: int) -> list[int] | None:
    """Lee 'k: sí/no' para k = 1..n. None si el formato no cuadra."""
    encontrados = {}
    for linea in (texto or "").splitlines():
        m = re.match(r"\s*(\d+)\s*[:.)-]\s*(s[ií]|no|yes)\b", _norm(linea))
        if m:
            encontrados[int(m.group(1))] = 0 if m.group(2) == "no" else 1
    if all(k in encontrados for k in range(1, n + 1)):
        return [encontrados[k] for k in range(1, n + 1)]
    return None


def _si_no(texto: str) -> int:
    t = _norm(texto).strip()
    return 0 if re.match(r"^\W*no\b", t) else int(bool(re.match(r"^\W*(si|yes)\b", t)))


def _lote_si_no(llm, system: str, encabezado: str, items: list[str], max_tokens_por_item: int = 6) -> list[int]:
    """Un solo prompt con todos los ítems; si el evaluador no respeta el formato,
    se cae a una llamada por ítem (más caro, pero siempre da un veredicto)."""
    if not items:
        return []
    lista = "\n".join(f"{i}. {t}" for i, t in enumerate(items, 1))
    user = (f"{encabezado}\n\n{lista}\n\nResponde una línea por ítem con el formato "
            f"'número: sí' o 'número: no'. Nada más.")
    v = veredictos(llm(system, user, max_tokens_por_item * len(items) + 8), len(items))
    if v is not None:
        return v
    return [_si_no(llm(system, f"{encabezado}\n\nÍtem: {t}\n\nResponde solo 'sí' o 'no'.", 4)) for t in items]


def average_precision(relevancias: list[int]) -> float | None:
    """Context precision de RAGAS: media de precision@k en las posiciones relevantes."""
    if not relevancias or not any(relevancias):
        return 0.0 if relevancias else None
    acumulado, suma = 0, 0.0
    for k, r in enumerate(relevancias, 1):
        acumulado += r
        if r:
            suma += acumulado / k
    return suma / sum(relevancias)


# --------------------------------------------------------------------------
# Las cuatro métricas (con evaluador LLM)
# --------------------------------------------------------------------------

SYSTEM_EVALUADOR = ("Eres un evaluador riguroso de un sistema de preguntas y respuestas en español. "
                    "Juzgas SOLO con la información que se te da.")


def faithfulness(llm, pregunta: str, respuesta: str, contextos: list[str], observaciones: list[str]) -> float | None:
    fuentes = [c for c in contextos if c] + [o for o in observaciones if o]
    if not fuentes:
        return None                      # sin nada observado, la métrica no aplica
    claims = afirmaciones(respuesta)
    if not claims:
        return None
    material = "\n\n".join(f"[Fuente {i}] {f}" for i, f in enumerate(fuentes, 1))
    encabezado = (f"PREGUNTA:\n{pregunta}\n\nFUENTES (documentos y resultados de herramientas):\n{material}\n\n"
                  "Para cada afirmación, ¿se puede deducir DIRECTAMENTE de las fuentes o de los datos de la "
                  "pregunta? Un cálculo cuyo resultado no aparece en las fuentes NO se puede deducir.")
    v = _lote_si_no(llm, SYSTEM_EVALUADOR, encabezado, claims)
    return sum(v) / len(v)


def context_precision(llm, pregunta: str, referencia: str, contextos: list[str]) -> tuple[float | None, list[int]]:
    if not contextos:
        return None, []
    encabezado = (f"PREGUNTA:\n{pregunta}\n\nRESPUESTA DE REFERENCIA:\n{referencia}\n\n"
                  "Para cada fragmento recuperado, ¿fue útil para llegar a la respuesta de referencia?")
    rel = _lote_si_no(llm, SYSTEM_EVALUADOR, encabezado, [c[:600] for c in contextos])
    return average_precision(rel), rel


def context_recall(llm, referencia: str, contextos: list[str]) -> float | None:
    if not contextos:
        return None
    frases = [f for f in afirmaciones(referencia) if not re.match(r"respuesta final", _norm(f))]
    if not frases:
        return None
    material = "\n\n".join(c[:600] for c in contextos)
    encabezado = (f"CONTEXTO RECUPERADO:\n{material}\n\n"
                  "Para cada frase de la respuesta de referencia, ¿la información que contiene "
                  "(el dato o la regla, no el cálculo) está en el contexto?")
    v = _lote_si_no(llm, SYSTEM_EVALUADOR, encabezado, frases)
    return sum(v) / len(v)


def answer_relevancy(llm, similitud, pregunta: str, respuesta: str, n: int = 3) -> float | None:
    if not (respuesta or "").strip():
        return 0.0
    salida = llm(SYSTEM_EVALUADOR,
                 f"RESPUESTA:\n{respuesta}\n\nEscribe {n} preguntas DISTINTAS que esta respuesta contestaría. "
                 "Una por línea, sin numerar ni explicar.", 40 * n)
    preguntas = [re.sub(r"^\s*[\d\-.•)]+\s*", "", l).strip() for l in salida.splitlines()]
    preguntas = [p for p in preguntas if len(p.split()) >= 3][:n]
    if not preguntas:
        return 0.0
    return sum(similitud(pregunta, p) for p in preguntas) / len(preguntas)


# --------------------------------------------------------------------------
# Versiones objetivas (para calibrar al evaluador)
# --------------------------------------------------------------------------

def faithfulness_numerica(pregunta: str, respuesta: str, contextos: list[str], observaciones: list[str]) -> float | None:
    texto = re.sub(r"Paso\s*\d+\s*:", " ", respuesta or "", flags=re.IGNORECASE)
    nums = numeros_en(texto)
    if not nums:
        return None
    disponibles = numeros_en(pregunta) + [n for c in contextos for n in numeros_en(c)] \
        + [n for o in observaciones for n in numeros_en(o)]
    # Los porcentajes de un documento aparecen como "40 por ciento" y se usan como 0.4.
    extendidos = disponibles + [str(a_numero(d) / 100) for d in disponibles if a_numero(d) is not None]
    respaldados = sum(1 for n in nums if any(mismo_numero(n, d) for d in extendidos))
    return respaldados / len(nums)


def relevancias_oro(contextos: list[str], clave: str | None) -> list[int]:
    if not clave:
        return []
    return [int(_norm(clave) in _norm(c)) for c in contextos]


def context_precision_oro(contextos: list[str], clave: str | None) -> float | None:
    if not contextos or not clave:
        return None
    return average_precision(relevancias_oro(contextos, clave))


def context_recall_oro(contextos: list[str], clave: str | None) -> float | None:
    if not clave:
        return None
    return float(any(relevancias_oro(contextos, clave))) if contextos else 0.0


# --------------------------------------------------------------------------
# Calibración
# --------------------------------------------------------------------------

def kappa_cohen(a: list[int], b: list[int]) -> float | None:
    if not a or len(a) != len(b):
        return None
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa, pb = sum(a) / n, sum(b) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)

---
## 5 · El evaluador

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Evaluador de RAGAS: Qwen2.5-7B-Instruct en 4 bits (~5.5 GB en la T4).
#
# Decisión: el juez de M2 (1.5B) tuvo kappa 0.286 contra la verdad numérica y
# comprimía 31 de 42 notas en un 3. RAGAS exige juicios más finos (¿esta frase se
# deduce de este pasaje?), así que usa un evaluador de 7B. El juez de M2 NO se toca:
# la Dimensión 2 del harness sigue siendo comparable con M2, S07 y S08.
EVALUADOR = "Qwen/Qwen2.5-7B-Instruct"     # si no cabe: "Qwen/Qwen2.5-3B-Instruct"

eval_tok = AutoTokenizer.from_pretrained(EVALUADOR)
eval_model = AutoModelForCausalLM.from_pretrained(
    EVALUADOR, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16),
).eval()
print("Evaluador:", EVALUADOR, "| memoria GPU:",
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB" if torch.cuda.is_available() else "cpu")

_CACHE_EVAL = {}


@torch.no_grad()
def evaluador(system, user, max_new_tokens=64):
    clave = (system, user, max_new_tokens)
    if clave not in _CACHE_EVAL:
        prompt = eval_tok.apply_chat_template([{"role": "system", "content": system},
                                               {"role": "user", "content": user}],
                                              tokenize=False, add_generation_prompt=True)
        ids = eval_tok(prompt, return_tensors="pt", add_special_tokens=False).to(eval_model.device)
        out = eval_model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=eval_tok.eos_token_id)
        _CACHE_EVAL[clave] = eval_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return _CACHE_EVAL[clave]


print(evaluador(SYSTEM_EVALUADOR, "Contexto: el examen vale 40%.\n\n1. El examen vale 40%.\n2. El examen vale 50%.\n\n"
                "¿Cada afirmación se deduce del contexto? Responde 'número: sí' o 'número: no'.", 20))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Evaluador: Qwen/Qwen2.5-7B-Instruct | memoria GPU: 5.9 GB
1: sí

2: no


In [13]:
from sentence_transformers import SentenceTransformer

st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


def similitud(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---
## 6 · Cálculo

In [14]:
# Qué recibe cada métrica, por configuración:
#   contextos     : chunks que el sistema tuvo delante. C2–C4: el contexto fijo;
#                   C5: los que devolvió buscar_documentos. C0/C1: ninguno.
#   observaciones : lo que devolvieron las herramientas (incluidos los errores, porque
#                   "no está definida" se apoya en el error de dividir(15, 0)).
# Aplicabilidad (declarada, no escondida):
#   faithfulness                   si hubo contextos u observaciones
#   context_precision / recall     solo en casos que requieren documentos, tienen clave de oro
#                                  y el sistema tuvo contexto (C0/C1 no tienen; C5 puede no buscar)
#   answer_relevancy               siempre
CONFIGS_RAGAS = NOMBRES              # para abreviar, p. ej. ["C1 · +LoRA", "C2 · +RAG", "C5 · agente"]
RUTA_RAGAS = Path("resultados/ragas_s10.jsonl")

HECHOS = {}
if RUTA_RAGAS.exists():
    for l in RUTA_RAGAS.read_text(encoding="utf-8").splitlines():
        f = json.loads(l)
        HECHOS[(f["config"], f["id"])] = f
print("Ya calculados:", len(HECHOS))


def texto_observaciones(reg):
    return [f"{o['herramienta']}({json.dumps(o['argumentos'], ensure_ascii=False)}) -> "
            f"{o['resultado'] if o['ok'] else 'ERROR: ' + str(o['error'])}"
            for o in reg["observaciones"] if o["herramienta"] != "buscar_documentos"]


def ragas_de(reg):
    cid, caso = reg["id"], CASOS[reg["id"]]
    ctx, obs = reg["contextos"], texto_observaciones(reg)
    clave = CLAVE_ORO.get(cid) if necesita_documentos(cid) else None
    fila = {"config": reg["config"], "id": cid, "bloque": caso["bloque"], "acierto": ACIERTO[(reg["config"], cid)],
            "n_contextos": len(ctx), "n_observaciones": len(obs)}
    fila["faithfulness"] = faithfulness(evaluador, caso["input"], reg["respuesta"], ctx, obs)
    fila["faithfulness_numerica"] = faithfulness_numerica(caso["input"], reg["respuesta"], ctx, obs)
    fila["answer_relevancy"] = answer_relevancy(evaluador, similitud, caso["input"], reg["respuesta"])
    if clave is not None and ctx:
        cp, rel = context_precision(evaluador, caso["input"], caso["esperado"], ctx)
        fila.update(context_precision=cp, relevancias_llm=rel, relevancias_oro=relevancias_oro(ctx, clave),
                    context_precision_oro=context_precision_oro(ctx, clave),
                    context_recall=context_recall(evaluador, caso["esperado"], ctx),
                    context_recall_oro=context_recall_oro(ctx, clave))
    return fila


pendientes = [r for r in respuestas if r["config"] in CONFIGS_RAGAS and (r["config"], r["id"]) not in HECHOS]
t0 = time.time()
for i, reg in enumerate(pendientes, 1):
    fila = ragas_de(reg)
    HECHOS[(reg["config"], reg["id"])] = fila
    with RUTA_RAGAS.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(fila, ensure_ascii=False, default=float) + "\n")
    if i % 25 == 0 or i == len(pendientes):
        print(f"  {i}/{len(pendientes)}  ({(time.time() - t0) / i:.1f} s por respuesta)")

df_ragas = pd.DataFrame([f for f in HECHOS.values() if f["config"] in CONFIGS_RAGAS])
print(df_ragas.shape)

Ya calculados: 0
  25/371  (0.9 s por respuesta)
  50/371  (0.9 s por respuesta)
  75/371  (0.9 s por respuesta)
  100/371  (0.9 s por respuesta)
  125/371  (1.0 s por respuesta)
  150/371  (1.0 s por respuesta)
  175/371  (1.1 s por respuesta)
  200/371  (1.0 s por respuesta)
  225/371  (1.1 s por respuesta)
  250/371  (1.1 s por respuesta)
  275/371  (1.1 s por respuesta)
  300/371  (1.1 s por respuesta)
  325/371  (1.1 s por respuesta)
  350/371  (1.1 s por respuesta)
  371/371  (1.1 s por respuesta)
(371, 15)


---
## 7 · Calibración del evaluador

Antes de leer una sola métrica: **¿el evaluador tiene razón?** Tres comprobaciones
contra las versiones objetivas.

In [15]:
from scipy.stats import spearmanr

print("CALIBRACIÓN DEL EVALUADOR CONTRA LAS VERSIONES OBJETIVAS")
print("=" * 78)

# 1 · Relevancia de cada chunk: evaluador vs. "el chunk contiene la clave de oro"
llm_rel, oro_rel = [], []
for f in df_ragas.dropna(subset=["context_precision_oro"]).itertuples():
    if isinstance(f.relevancias_llm, list) and len(f.relevancias_llm) == len(f.relevancias_oro):
        llm_rel += f.relevancias_llm
        oro_rel += f.relevancias_oro
k_rel = kappa_cohen(llm_rel, oro_rel)
print(f"Relevancia de chunks ({len(oro_rel)} juicios): acuerdo "
      f"{np.mean(np.array(llm_rel) == np.array(oro_rel)):.1%} | kappa {k_rel:.3f}" if oro_rel else "sin datos")

# 2 · Context recall (binarizado en 0.5) vs. "la clave está en algún chunk"
cr = df_ragas.dropna(subset=["context_recall", "context_recall_oro"])
k_rec = kappa_cohen([int(v >= 0.5) for v in cr["context_recall"]], [int(v) for v in cr["context_recall_oro"]]) if len(cr) else None
print(f"Context recall ({len(cr)} casos): kappa {k_rec:.3f}" if k_rec is not None else "Context recall: sin datos")

# 3 · Faithfulness del evaluador vs. la numérica (¿los números salen de lo observado?)
ff = df_ragas.dropna(subset=["faithfulness", "faithfulness_numerica"])
rho = spearmanr(ff["faithfulness"], ff["faithfulness_numerica"]).correlation if len(ff) > 2 else None
print(f"Faithfulness LLM vs numérica ({len(ff)} respuestas): Spearman {rho:.3f}" if rho is not None else "sin datos")

CALIBRACION = {"kappa_relevancia_chunks": k_rel, "n_juicios_relevancia": len(oro_rel),
               "kappa_context_recall": k_rec, "spearman_faithfulness": rho}
print("=" * 78)
print("Lectura (S06): kappa >= 0.6 aceptable, >= 0.8 fuerte. Si el evaluador no llega,")
print("las conclusiones de contexto se apoyan en las versiones _oro, que son objetivas.")

CALIBRACIÓN DEL EVALUADOR CONTRA LAS VERSIONES OBJETIVAS
Relevancia de chunks (153 juicios): acuerdo 86.3% | kappa 0.720
Context recall (51 casos): kappa 0.000
Faithfulness LLM vs numérica (198 respuestas): Spearman 0.093
Lectura (S06): kappa >= 0.6 aceptable, >= 0.8 fuerte. Si el evaluador no llega,
las conclusiones de contexto se apoyan en las versiones _oro, que son objetivas.


---
## 8 · RAGAS por configuración

In [16]:
METRICAS_RAGAS = ["faithfulness", "faithfulness_numerica", "context_precision", "context_precision_oro",
                  "context_recall", "context_recall_oro", "answer_relevancy"]

tabla = (df_ragas.groupby("config", sort=False)[METRICAS_RAGAS]
         .agg(lambda s: s.dropna().mean() if s.notna().any() else np.nan)
         .reindex([n for n in NOMBRES if n in CONFIGS_RAGAS]))
cobertura = df_ragas.groupby("config", sort=False)[METRICAS_RAGAS].agg(lambda s: s.notna().sum())
print("RAGAS por configuración (media sobre los casos donde la métrica aplica)")
print(tabla.round(3).to_string())
print("\nn por métrica:")
print(cobertura.reindex(tabla.index).to_string())

RAGAS por configuración (media sobre los casos donde la métrica aplica)
                   faithfulness  faithfulness_numerica  context_precision  context_precision_oro  context_recall  context_recall_oro  answer_relevancy
config                                                                                                                                                
C0 · base                   NaN                  0.357                NaN                    NaN             NaN                 NaN             0.678
C1 · +LoRA                  NaN                  0.323                NaN                    NaN             NaN                 NaN             0.709
C2 · +RAG                 0.651                  0.448                1.0                    0.9           0.689                 1.0             0.671
C3 · +herram.             0.687                  0.484                1.0                    0.9           0.689                 1.0             0.650
C4 · +ReAct           

> **Cómo leerlo (S10).**
> - **context precision/recall bajas** → problema de retrieval. Hay que volver a S08.
> - **faithfulness baja** → el modelo no se apoya en lo que observó: ajustar el prompt
>   o quitar ruido.
> - **answer relevancy baja** → la respuesta divaga.
>
> En C0 y C1 la faithfulness no aplica (no observan nada), y la de contexto solo
> aplica en C2–C5.

### 8.1 · Aciertos por la razón equivocada

In [17]:
# La alerta de S10: "una respuesta puede acertar por la razón equivocada".
# Aciertos con fidelidad numérica < 1: el número final es correcto, pero algún número
# de la respuesta no sale de la pregunta, de los documentos ni de las herramientas
# (se calculó "de cabeza").
alerta = df_ragas[(df_ragas["acierto"]) & (df_ragas["faithfulness_numerica"].notna())]
resumen_alerta = alerta.groupby("config", sort=False).apply(
    lambda g: pd.Series({"aciertos_con_numeros": len(g),
                         "respaldados_del_todo": int((g["faithfulness_numerica"] >= 0.999).sum()),
                         "con_calculo_de_cabeza": int((g["faithfulness_numerica"] < 0.999).sum())}))
print(resumen_alerta.reindex([n for n in NOMBRES if n in resumen_alerta.index]).to_string())

                   aciertos_con_numeros  respaldados_del_todo  con_calculo_de_cabeza
config                                                                              
C0 · base                            18                     1                     17
C1 · +LoRA                           24                     0                     24
C2 · +RAG                            24                     3                     21
C3 · +herram.                        28                     3                     25
C4 · +ReAct                          24                     4                     20
C5 · agente                          26                     5                     21
C5-FT · entrenado                    29                    14                     15


/tmp/ipykernel_1745/1620524886.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resumen_alerta = alerta.groupby("config", sort=False).apply(


### 8.2 · Fallos de conocimiento: ¿retrieval o generación?

Es el mismo diagnóstico de S07 §11, ahora para todas las configuraciones y con
recall de oro. En C5 aparece una causa nueva: **no buscó** cuando hacía falta, que es
un fallo de la **decisión** del agente, no del retrieval.

In [18]:
# Harness × RAGAS en los casos que requieren documentos: ¿falló el retrieval o la generación?
def causa(f):
    if f["acierto"]:
        return "acierto"
    if f["n_contextos"] == 0:
        return "SIN CONTEXTO: no tiene RAG o no buscó"
    if f["context_recall_oro"] == 0:
        return "RETRIEVAL: el dato no llegó"
    return "GENERACIÓN: el dato llegó y no lo usó"


docs = df_ragas[df_ragas["id"].map(necesita_documentos)].copy()
docs["causa"] = docs.apply(causa, axis=1)
print(pd.crosstab(docs["config"], docs["causa"]).reindex([n for n in NOMBRES if n in set(docs["config"])]).to_string())

causa              GENERACIÓN: el dato llegó y no lo usó  SIN CONTEXTO: no tiene RAG o no buscó  acierto
config                                                                                                  
C0 · base                                              0                                     15        0
C1 · +LoRA                                             0                                     15        0
C2 · +RAG                                              8                                      0        7
C3 · +herram.                                          6                                      0        9
C4 · +ReAct                                            6                                      0        9
C5 · agente                                            1                                      4       10
C5-FT · entrenado                                      1                                      8        6


---
## 9 · El scorecard final de M3

Harness de M2 + eval set M3 + herramientas + proceso + robustez (notebook 3) +
RAGAS (este notebook), en una tabla.

In [19]:
# Scorecard final de la entrega M3: el scorecard del agente (notebook 3) + RAGAS.
sc_agente = pd.read_csv("resultados/scorecard_agente.csv").set_index("metrica")
filas_ragas = pd.DataFrame({
    "RAGAS · faithfulness": tabla["faithfulness"],
    "RAGAS · faithfulness numérica (objetiva)": tabla["faithfulness_numerica"],
    "RAGAS · context precision": tabla["context_precision"],
    "RAGAS · context precision (oro)": tabla["context_precision_oro"],
    "RAGAS · context recall": tabla["context_recall"],
    "RAGAS · context recall (oro)": tabla["context_recall_oro"],
    "RAGAS · answer relevancy": tabla["answer_relevancy"],
}).T
scorecard_final = pd.concat([sc_agente[[c for c in sc_agente.columns if c in filas_ragas.columns]], filas_ragas])
scorecard_final.index.name = "metrica"
pd.set_option("display.width", 200)
print(scorecard_final.apply(pd.to_numeric, errors="coerce").round(3).to_string())

                                          C0 · base  C1 · +LoRA  C2 · +RAG  C3 · +herram.  C4 · +ReAct  C5 · agente  C5-FT · entrenado
metrica                                                                                                                               
Exactitud en cálculo                          0.500       0.833      0.583          0.583        0.583        0.583              0.500
Aciertos totales /21                          9.000      13.000     13.000         13.000       14.000       12.000             13.000
Conocimiento /4                               0.000       0.000      3.000          3.000        3.000        3.000              3.000
Adversarial /5                                3.000       3.000      3.000          3.000        4.000        2.000              4.000
Alucinaciones                                12.000       8.000      5.000          8.000        7.000        9.000              8.000
Abstenciones                                  0.000    

### 9.1 · Weights & Biases (opcional)

Una corrida por configuración, la tabla comparativa y la traza del agente en un
caso compuesto. Corre **offline** sin cuenta, como en el extra de S10.

In [20]:
# Weights & Biases (extra de S10): una corrida por configuración + tabla comparativa + traza.
# Offline por defecto: no pide cuenta. Para subirlo: `wandb sync wandb/offline-run-*`.
USAR_WANDB = True
if USAR_WANDB:
    import wandb
    os.environ.setdefault("WANDB_MODE", "offline")
    PROYECTO_W = "tutor-matematicas-arquitecturas"
    numerico = scorecard_final.apply(pd.to_numeric, errors="coerce")
    for n in numerico.columns:
        run = wandb.init(project=PROYECTO_W, name=f"s10-{n}", group="s10-agente",
                         config={"configuracion": n, "evaluador_ragas": EVALUADOR}, reinit=True)
        wandb.log({k: v for k, v in numerico[n].dropna().items()})
        wandb.finish()

    run = wandb.init(project=PROYECTO_W, name="s10-comparativa", group="s10-agente", reinit=True)
    t = wandb.Table(dataframe=numerico.reset_index())
    wandb.log({"scorecard_final": t, "calibracion": CALIBRACION})
    # La traza de C5 en un caso compuesto: qué pensó, qué llamó y qué observó.
    reg = next((r for r in respuestas if r["config"].startswith("C5 ") and r["id"] == "com-01"), None)
    if reg:
        traza = wandb.Table(columns=["paso", "tipo", "datos"])
        for e in reg["traza"]:
            traza.add_data(e["paso"], e["tipo"], json.dumps(e["datos"], ensure_ascii=False, default=str)[:500])
        wandb.log({"traza_c5_com01": traza})
    wandb.finish()
    print("Corridas registradas en W&B (modo:", os.environ["WANDB_MODE"], ")")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+16,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+16,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+21,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+24,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+21,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+24,...


Abstenciones,▁
Aciertos totales /21,▁
Adversarial /5,▁
Alucinaciones,▁
Alucinaciones M3,▁
Aritmética /13,▁
Compuesto /8,▁
Conocimiento /4,▁
Errores de formato,▁
Exactitud en cálculo,▁
+24,...


Corridas registradas en W&B (modo: offline )


### 9.2 · Referencia: la librería `ragas`

Para quien quiera contrastar con la implementación oficial (necesita un LLM y
embeddings; por defecto, OpenAI):

```python
# %pip install -q ragas datasets
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
# filas = [r for r in respuestas if r["config"] == "C5 · agente"]
# ds = Dataset.from_dict({
#     "question":     [CASOS[r["id"]]["input"] for r in filas],
#     "answer":       [r["respuesta"] for r in filas],
#     "contexts":     [r["contextos"] + texto_observaciones(r) for r in filas],
#     "ground_truth": [CASOS[r["id"]]["esperado"] for r in filas],
# })
# evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
```

---
## 10 · Artefactos

In [21]:
scorecard_final.to_csv("resultados/scorecard_final_m3.csv", encoding="utf-8")
tabla.to_csv("resultados/ragas_scorecard_s10.csv", encoding="utf-8")
Path("resultados/ragas_s10.json").write_text(json.dumps({
    "evaluador": EVALUADOR,
    "calibracion": CALIBRACION,
    "ragas_por_config": json.loads(tabla.to_json(orient="index")),
    "cobertura": json.loads(cobertura.to_json(orient="index")),
    "diagnostico_documentos": json.loads(pd.crosstab(docs["config"], docs["causa"]).to_json(orient="index")),
}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
for p in ["ragas_s10.jsonl", "ragas_scorecard_s10.csv", "ragas_s10.json", "scorecard_final_m3.csv"]:
    print(f"  resultados/{p}")

  resultados/ragas_s10.jsonl
  resultados/ragas_scorecard_s10.csv
  resultados/ragas_s10.json
  resultados/scorecard_final_m3.csv


---
## 11 · Discusión: la lectura honesta que pide la entrega

**1. ¿Qué técnica movió qué?** Crucen los deltas del notebook 3 con este scorecard.
Por ejemplo: si C5 sube en *compuesto* y su context recall (oro) es alto, el agente
**decidió buscar** y el retrieval respondió. Si sube la faithfulness numérica de C1 a
C3, las herramientas hicieron que los números salgan de algún sitio verificable.

**2. ¿Qué costó?** Tokens y segundos por pregunta (notebook 3). Un agente que gana
pocos casos y multiplica el coste se reporta como tal.

**3. ¿Qué falla queda pendiente?** Los *fallos por generación* de 8.2, los *aciertos
con cálculo de cabeza* de 8.1 y lo que diga la calibración sobre el propio evaluador.

**4. ¿Se justifica el agente?** Solo si el harness mejora **y** RAGAS confirma que
mejora por la razón correcta. Si mejora el harness pero cae la faithfulness, acertó
por suerte; si mejora RAGAS pero no el harness, el tutor no es mejor para el
estudiante.

---
## 12 · Conclusiones de esta corrida

| | C0 | C1 | C2 | C3 | C4 | C5 | C5-FT |
|---|---|---|---|---|---|---|---|
| faithfulness (7B) | — | — | 0.65 | 0.69 | 0.69 | 0.77 † | 0.73 |
| faithfulness numérica (objetiva) | 0.36 | 0.32 | 0.45 | 0.48 | 0.48 | 0.40 | **0.77** |
| context precision (oro) | — | — | 0.90 | 0.90 | 0.90 | **1.00** | **1.00** |
| context recall (oro) | — | — | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| answer relevancy | 0.68 | **0.71** | 0.67 | 0.65 | 0.67 | 0.70 | 0.66 |

† n = 10: C5 casi nunca observó nada, así que su faithfulness no es comparable con
las demás columnas, medidas sobre 48–53 casos. La cobertura está en la tabla de `n`.

**1 · Primero la calibración, porque condiciona todo lo demás.** Kappa 0.72 en
relevancia de chunks: aceptable según el criterio de S06. Pero Spearman 0.09 entre la
faithfulness del 7B y la numérica objetiva, y kappa 0 en context recall porque las 15
consultas documentales tienen recall de oro = 1 y no hay varianza que medir. Por eso
las conclusiones de fundamento se apoyan en la **faithfulness numérica**, no en el
juicio del evaluador.

**2 · La alerta de S10 se confirma, y las herramientas la corrigen.** De los aciertos
de C1, **0 de 24** tienen todos sus números respaldados por la pregunta, un documento
o una herramienta: el tutor sin herramientas acierta calculando de cabeza. En C5-FT
son **14 de 29**, y su faithfulness numérica es la más alta (0.77) frente al 0.32 de
C1, que es la más baja de todas. Es la evidencia más clara de que las herramientas
cambian la *naturaleza* del acierto, no solo su cantidad.

**3 · El retrieval no falla nunca.** Context recall de oro = 1.00 en todas las
configuraciones con contexto, y precision de oro 0.90–1.00. Coincide con S08: el
cuello de botella nunca estuvo en recuperar.

**4 · Lo que cambió es la naturaleza del fallo.** En C2–C4 los fallos de conocimiento
son 6–8 casos de *"el dato llegó y el modelo no lo usó"* (generación). En C5 y C5-FT
son 4 y 8 casos de *"no buscó"* (decisión del agente). Dar autonomía no eliminó el
fallo: lo movió de la generación a la decisión, y en C5-FT lo empeoró.

**5 · El evaluador sobreestima la relevancia del contexto.** Da context precision 1.00
en todas las configuraciones, mientras la versión objetiva marca 0.90 en C2–C4. Es
consistente con el sesgo hacia el "sí" que ya se le vio al juez de 1.5B en M2, ahora
medido en un modelo cinco veces mayor.

**6 · RAGAS y el harness dicen cosas distintas, y por eso se reportan juntos.** El
harness dice que C1 es tan bueno como C5-FT en el eval set de M2; RAGAS dice que C1
llega ahí sin apoyarse en nada verificable. Un tutor que acierta de memoria es más
frágil en producción que uno que consulta y calcula, aunque el scorecard los empate.